# Creation of artifacts

In [0]:
from pyspark.sql.types import *
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.ml.linalg import VectorUDT


In [0]:
%run ./variables

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()  
target_path = "/Shared/pe_memberdna"
w.workspace.mkdirs(target_path)

## Schemas

In [0]:
# This path is configured depending on the environment we are deploying assets into.
#environment = 'dev' # should be ingested as variable from the run ./variables
managed_location_base_path = f""" "s3://dbx-edp{environment}-ds-enterprise-pe-uc-storage/pe" """

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema_name}
MANAGED LOCATION {managed_location_base_path}
""")

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema_name}
MANAGED LOCATION {managed_location_base_path}
""")

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{pe_schema_name}
MANAGED LOCATION {managed_location_base_path}
""")

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema_name}
MANAGED LOCATION {managed_location_base_path}
""")

## Volumes

In [0]:
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {catalog_name}.{silver_schema_name}.helpers
          """)

In [0]:
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {catalog_name}.{pe_schema_name}.helpers
          """)

In [0]:
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {catalog_name}.{pe_schema_name}.outputs_for_s3
          """)

In [0]:
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {catalog_name}.{pe_schema_name}.job_summary_reports          
          """)

In [0]:
dbutils.fs.mkdirs(f"/Volumes/{catalog_name}/{silver_schema_name}/helpers/STATS/ETL/archive/")

In [0]:
dbutils.fs.mkdirs(f"/Volumes/{catalog_name}/{pe_schema_name}/helpers/trip_propensity/")
dbutils.fs.mkdirs(f"/Volumes/{catalog_name}/{pe_schema_name}/helpers/trip_spend/")
dbutils.fs.mkdirs(f"/Volumes/{catalog_name}/{pe_schema_name}/helpers/bbm_propensity_model/")
dbutils.fs.mkdirs(f"/Volumes/{catalog_name}/{pe_schema_name}/helpers/digital_propensity/")
dbutils.fs.mkdirs(f'/Volumes/{catalog_name}/{pe_schema_name}/outputs_for_s3/ASSIGNMENTS/campaigns/FY26/MMPC18FY26/Coupon Input/')
dbutils.fs.mkdirs(f'/Volumes/{catalog_name}/{pe_schema_name}/outputs_for_s3/ASSIGNMENTS/cdsa')
dbutils.fs.mkdirs(dna_cube_volume_path)
dbutils.fs.mkdirs(bbm_score_volume_path)

## Tables

### Bronze

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_transaction_detail} (
    PURCH_HDR_ID STRING,
    PURCH_DTL_ID STRING,
    PURCH_DT STRING,
    GTIN_CD STRING,
    ARTICLE_NBR STRING,
    MC_CD STRING,
    EXTENDED_PRC_AMT STRING,
    EXTENDED_UNIT_PRC_AMT STRING,
    SALES_QTY STRING,
    SALES_UOM STRING,
    QTY_IN_UNITS STRING,
    NORMAL_PRC_AMT STRING,
    NORMAL_UNIT_PRC_AMT STRING,
    REDUCTION_AMT STRING,
    SCANNED_VS_KEYED_IND STRING,
    DISCOUNT_TYPE_CD STRING,
    DISCOUNT_PURCH_DTL_ID STRING,
    VOIDED_FLAG STRING,
    VOIDED_PURCH_DTL_ID STRING,
    RSN_CD STRING,
    SALES_CTGRY_CD STRING,
    RETURN_IND STRING,
    REBATE_IND STRING,
    OFFER_ID STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_master_item} (
    GTIN_CD STRING,
    ARTICLE_DESC STRING,
    ARTICLE_NBR STRING,
    MCH4_CD STRING,
    MCH4_DESC STRING,
    MCH3_CD STRING,
    MCH3_DESC STRING,
    MCH2_CD STRING,
    MCH2_DESC STRING,
    MCH1_CD STRING,
    MCH1_DESC STRING,
    MC_CD STRING,
    MC_DESC STRING,
    AH1_CD STRING,
    AH1_DESC STRING,
    AH2_CD STRING,
    AH2_DESC STRING,
    AH3_CD STRING,
    AH3_DESC STRING,
    AH4_CD STRING,
    AH4_DESC STRING,
    AH5_CD STRING,
    AH5_DESC STRING,
    AH6_CD STRING,
    AH6_DESC STRING,
    BRAND_TYPE STRING,
    EFF_DT STRING,
    EXP_DT STRING,
    ORDR_AS STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_master_brand} (
    _c0 STRING,
    _c1 STRING,
    _c2 STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_master_member} (
    _c0 STRING,
    _c1 STRING,
    _c2 STRING,
    _c3 STRING,
    _c4 STRING,
    _c5 STRING,
    _c6 STRING,
    _c7 STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_master_member_history} (
    MBRSHP_HIST_SID STRING,
    MBRSHP_SID STRING,
    EFF_DT STRING,
    EXP_DT STRING,
    MBRSHP_STAT_CD STRING,
    MBRSHP_EXP_DT STRING,
    MBRSHP_RNWL_DT STRING,
    MBRSHP_FEE_INC STRING,
    RWDS_MBR_IND STRING,
    RWDS_MBR_ENR_DT STRING,
    CLUB_OF_FREQUENCY STRING,
    TEAM_MBR_IND STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_master_member_extended} (
    MBRSHP_NBR STRING,
    MBRSHP_SID STRING,
    MBRSHP_TYPE_ID STRING,
    MBRSHP_FEE_INC STRING,
    MBRSHP_SUB_TYPE STRING,
    MBRSHP_ENR_DT STRING,
    MBRSHP_EXP_DT STRING,
    MBRSHP_RNWL_DT STRING,
    RWDS_MBR_IND STRING,
    RWDS_MBR_ENR_DT STRING,
    CLUB_OF_FREQUENCY STRING,
    MKT_CD STRING,
    AUTO_RNWL_IND STRING,
    ER_SIGNUP_DT STRING,
    HOME_ZIP_CD STRING,
    SIC_CD STRING,
    GRP_AFFIL_ID STRING,
    HH_SID STRING,
    PRI_SUPP_FHH_IND STRING,
    MBR_STATE STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_transaction_header} (
    PURCH_HDR_ID STRING,
    MBRSHP_SID STRING,
    SITE_NBR STRING,
    PURCH_DT STRING,
    SALES_CHANNEL_ID STRING,
    TOT_SALES_AMT STRING,
    TAX_AMT STRING,
    PURCHASE_TM STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_transaction_payment} (
    PURCH_HDR_ID STRING,
    PURCH_PYMT_SEQ_ID STRING,
    MBRSHP_SID STRING,
    TENDER_TYPE_CD STRING,
    CPN_NBR STRING,
    PYMT_SCANNED_OR_KEYED_IND STRING,
    SALES_PYMT_AMT STRING,
    TENDER_ID STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_awards} (
    MBRSHP_SID STRING,
    AWRD_CERT_NBR STRING,
    AWRD_CERT_AMT STRING,
    AWRD_CERT_ISSUE_DT STRING,
    AWRD_CERT_EXP_DT STRING,
    AWRD_CERT_RDMPTN_CD STRING,
    AWRD_PROMO_ID STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_quotient_id} (
    LOYALTYNUMBER STRING,
    USERCODE STRING,
    SIGNUP DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_master_email} (
    MBRSHP_NBR STRING,
    MAIL_ID STRING,
    EMAIL_SUBJECT STRING,
    EMAIL_NAME STRING,
    FIRST_BOUNCE_DATE DATE,
    LAST_BOUNCE_DATE DATE,
    BOUNCE_TOTAL INTEGER,
    FIRST_SEND_DATE DATE,
    LAST_SEND_DATE DATE,
    SEND_TOTAL INTEGER,
    FIRST_OPEN_DATE DATE,
    LAST_OPEN_DATE DATE,
    OPEN_TOTAL INTEGER,
    FIRST_CLICK_DATE DATE,
    LAST_CLICK_DATE DATE,
    CLICK_TOTAL INTEGER,
    FIRST_UNSUB_DATE DATE,
    LAST_UNSUB_DATE DATE,
    UNSUB_TOTAL INTEGER
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_member_geo_archive} (
    merkle_id STRING,
    mbr_sid STRING,
    individual_id STRING,
    household_id STRING,
    address_id STRING,
    zip STRING,
    dsf_confirm_flag STRING,
    dsf_cmra_flag STRING,
    dsf_delivery_type STRING,
    dsf_residence STRING,
    dsf_business STRING,
    dsf_drop_flag STRING,
    dsf_drop_count STRING,
    dsf_throwback STRING,
    dsf_seasonal STRING,
    dsf_vacant STRING,
    fips_state_code STRING,
    fips_country_code STRING,
    usps_address_type STRING,
    apartment_flag STRING,
    time_zone STRING,
    dma_code STRING,
    geo_census_2010_tract STRING,
    geo_census_2010_block_group STRING,
    geo_msa_code STRING,
    geo_lat_or_long_level STRING,
    geo_latitude DECIMAL(10,6),
    geo_longitude DECIMAL(10,6),
    date STRING,
    day INTEGER,
    month INTEGER,
    year INTEGER,
    file_modification_time TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_member_tract} (
    MBRSHP_SID STRING,
    MEMBERID STRING,
    MEMTYPE STRING,
    CLUSTER STRING,
    ZIP STRING,
    TRACT STRING,
    LONGITUDE STRING,
    LATITUDE STRING,
    UPDATE STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_distance} (
    TRACT STRING,
    site_nbr STRING,
    Banner STRING,
    site_name STRING,
    addr_line_1 STRING,
    city_name STRING,
    state_cd STRING,
    zip_cd STRING,
    BJS_Longitude STRING,
    BJS_Latitude STRING,
    SQFT STRING,
    Gas_Opening STRING,
    Club_Opening STRING,
    BJS_TRACT_LON STRING,
    BJS_TRACT_LAT STRING,
    BJS_Driving_Distance STRING,
    BJS_Distance STRING,
    BJS_Drive_Time DOUBLE,
    TRACT_AREA STRING,
    WALMART_Driving_Distance STRING,
    WALMART_Drive_Time DOUBLE,
    WALMART_Distance STRING,
    COSTCO_Driving_Distance STRING,
    COSTCO_Drive_Time DOUBLE,
    COSTCO_Distance STRING,
    SAMS_Driving_Distance STRING,
    SAMS_Drive_Time DOUBLE,
    SAMS_Distance STRING,
    Zip_Distance STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_coupon_clip} (
    usercode STRING,
    EVENTDATETIME DATE,
    EVENTTYPE STRING,
    OFFERID STRING,
    STORENAME STRING,
    OFFERCODE STRING,
    OFFERACTIVEDATE DATE,
    OFFERSHUTOFFDATE DATE,
    OFFEREXPIRYDATE DATE,
    BRAND STRING,
    OFFERVALUE DOUBLE,
    OFFERTYPE STRING,
    DISCOUNTTYPE STRING,
    CATEGORY STRING,
    SOURCE STRING,
    APPID STRING,
    APPCODE STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_coupon_clip_usercode} (
    identifier STRING,
    usercode STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_master_club_with_brand} (
    _c0 STRING,
    _c1 STRING,
    _c2 STRING,
    _c3 STRING,
    _c4 STRING,
    _c5 STRING,
    _c6 STRING,
    _c7 STRING,
    _c8 STRING,
    _c9 STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

# if you get to use proper schema in the future
# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS {bronze_master_club_with_brand} (
#     SITE_NBR STRING,
#     SITE_NAME_2 STRING,
#     ADDR_LINE_2 STRING,
#     CITY_NAME STRING,
#     STATE_CD STRING,
#     ZIP_CD STRING,
#     ZN_NBR STRING,
#     RGN_NBR STRING,
#     SITE_TYPE STRING,
#     COMP_STTS STRING
# )

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_master_member_basic} (
    mbr_sid STRING,
    mbr_prmry_sid STRING,
    hh_rollup_prmry_sid STRING,
    mbr_stts_cd STRING,
    mbr_mkt_cd STRING,
    mbr_bsn_sic_cd STRING,
    mbr_bsn_sic_frst4_cd STRING,
    mbr_bsn_sic_lst4_cd STRING,
    mbr_grp_affilatn_cd STRING,
    mbr_rwd_typ STRING,
    mbr_clb_of_mbrshp STRING,
    mbr_typ_cd STRING,
    mbr_clb_usr_lst_updt_id STRING,
    mbr_ic_spplmntl_cnt STRING,
    mbr_upgrd_prmpt_cnt STRING,
    mbr_fuel_prmtn_elgbl_ind STRING,
    mbr_chk_stts_cd STRING,
    mbr_chk_fscl_ytd_cnt STRING,
    mbr_lst_updt_dt STRING,
    mbr_lst_updt_clb_nbr STRING,
    mbr_lst_sls_txn_dt STRING,
    mbr_enrl_dt STRING,
    mbr_exp_dt STRING,
    mbr_prev_exp_dt STRING,
    mbr_prorate_mth STRING,
    mbr_rnwl_dt STRING,
    mbr_mfi_amt STRING,
    mbr_rnwl_clb_nbr STRING,
    mbr_aqustn_prmtn_cd STRING,
    mbr_prescrn_id STRING,
    mbr_lst_prescrn_dt STRING,
    mbr_prescrn_exp_dt STRING,
    mbr_prescrn_src_ind STRING,
    mbr_pos_upgrd_prmpt_dt STRING,
    mbr_ezr_ask_dt STRING,
    mbr_ezr_signup_dt STRING,
    mbr_ezr_optout_dt STRING,
    mbr_addr_updt_dt STRING,
    mbr_addr_updt_channel_cd STRING,
    mbr_email_valid_ind STRING,
    mbr_email_frqncy_ind STRING,
    mbr_email_enrl_via_net STRING,
    mbr_email_rnwl_via_net STRING,
    mbr_email_upgrd_via_net STRING,
    mbr_email_addr_updt_channel_cd STRING,
    mbr_email_addr_updt_clb STRING,
    mbr_email_addr_updt_dt STRING,
    mbr_email_optin_prefr_updt_dt STRING,
    mbr_email_optin_prefr_updt_clb STRING,
    mbr_email_optin_prefr_updt_src STRING,
    mbr_dlvry_addr_ind STRING,
    mbr_tele_mrktng_optin_ind STRING,
    mbr_prmtn_optin_ind STRING,
    mbr_ezr_stts_cd STRING,
    mbr_ezr_stts_ind STRING,
    mbr_profile_ind STRING,
    mbr_household_ind STRING,
    mbr_tmbr_ind STRING,
    trial_mbr_ind STRING,
    mbr_do_not_offer_email_ind STRING,
    mbr_cell_txt_msg_opt_ind STRING,
    mbr_cell_txt_opt_chg_dt STRING,
    mbr_cell_txt_opt_chg_clb STRING,
    mbr_cell_txt_opt_chg_src_cd STRING,
    mbr_household_cnvrtd_ind STRING,
    mbr_free_sup_ind STRING,
    mbr_awrd_shrd_ind STRING,
    mbr_awrd_shrd_chg_dt STRING,
    mbr_scan_go_elgbl_ind STRING,
    mbr_phone_typ_cd STRING,
    mbr_household_cnvrtd_prnt_dt STRING,
    mbr_prtnr_sid STRING,
    mbr_arinfo_flg STRING,
    mbr_prmry_flg STRING,
    mbr_online_only_flg STRING,
    mbr_rwd_enrl_dt STRING,
    mbr_rwd_downgrade_to_basic_dt STRING,
    mbr_rwd_stts_change_dt STRING,
    mbr_rwd_change_to_stts_cd STRING,
    mbr_rwd_cobrand_enroll_dt STRING,
    ads_acct_stts_ind STRING,
    ads_acct_open_dt STRING,
    ads_acct_close_dt STRING,
    drvd_clb_of_frqncy STRING,
    drvd_clb_of_frqncy_rollup STRING,
    drvd_mktble_univ_cd STRING,
    drvd_mbr_sub_type_cd STRING,
    drvd_tenure_mth STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_bcg_maps_ah5_customer_facing_desc} (
    AH5_CD STRING,
    AH5_DESC STRING,
    MBR_FACING_CATEGORY STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_control_files_tab_01_redshift} (
    purch_dt STRING,
    tot_sales STRING,
    tot_trips STRING,
    tot_dtls STRING,
    tot_non_gas_sales STRING,
    tot_non_gas_trips STRING,
    tot_non_gas_dtls STRING,
    tot_gas_sales STRING,
    tot_gas_trips STRING,
    tot_gas_dtls STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_control_files_tab_02_redshift} (
    site_nbr STRING,
    purch_dt STRING,
    tot_sales STRING,
    tot_trips STRING,
    tot_dtls STRING,
    tot_non_gas_sales STRING,
    tot_non_gas_trips STRING,
    tot_non_gas_dtls STRING,
    tot_gas_sales STRING,
    tot_gas_trips STRING,
    tot_gas_dtls STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_control_files_tab_03_redshift} (
    mbrshp_sid STRING,
    tot_sales STRING,
    tot_trips STRING,
    tot_dtls STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_control_files_tab_04_redshift} (
    purch_hdr_id STRING,
    purch_dtl_id STRING,
    purch_dt STRING,
    gtin_cd STRING,
    article_nbr STRING,
    mc_cd STRING,
    extended_prc_amt STRING,
    sales_qty STRING,
    sales_uom STRING,
    extended_unit_prc_amt STRING,
    normal_prc_amt STRING,
    normal_unit_prc_amt STRING,
    reduction_amt STRING,
    tax_exempt_ind STRING,
    scanned_vs_keyed_ind STRING,
    retail_type_cd STRING,
    discount_type_cd STRING,
    discount_purch_dtl_id STRING,
    voided_flag STRING,
    voided_purch_dtl_id STRING,
    prmtn_nbr STRING,
    prmtn_cd STRING,
    fee_type_cd STRING,
    fee_type_amt STRING,
    rsn_cd STRING,
    serial_nbr STRING,
    sales_ctgry_cd STRING,
    return_ind STRING,
    rebate_ind STRING,
    prc_override_ind STRING,
    qty_in_units STRING,
    mdse_hier_sid STRING,
    article_hier_sid STRING,
    article_site_xref_sid STRING,
    mbrshp_nbr STRING,
    site_nbr STRING,
    transaction_type_cd STRING,
    register_nbr STRING,
    transaction_nbr STRING,
    sales_channel_id STRING,
    purchase_tm STRING,
    cashier STRING,
    mbr_crd_keyed_vs_scanned_ind STRING,
    bonusbuy_id STRING,
    discount_id STRING,
    offer_id STRING,
    insert_by STRING,
    insert_ts STRING,
    lst_upd_by STRING,
    lst_upd_ts STRING,
    insert_load_nbr STRING,
    lst_upd_load_nbr STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_control_files_tab_05_redshift} (
    purch_dt STRING,
    tot_sales STRING,
    tot_trips STRING,
    tot_pymts STRING,
    tot_non_gas_sales STRING,
    tot_non_gas_trips STRING,
    tot_non_gas_pymts STRING,
    tot_gas_sales STRING,
    tot_gas_trips STRING,
    tot_gas_pymts STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_control_files_tab_06_redshift} (
    site_nbr STRING,
    purch_dt STRING,
    tot_sales STRING,
    tot_trips STRING,
    tot_pymts STRING,
    tot_non_gas_sales STRING,
    tot_non_gas_trips STRING,
    tot_non_gas_pymts STRING,
    tot_gas_sales STRING,
    tot_gas_trips STRING,
    tot_gas_pymts STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_control_files_tab_07_redshift} (
    purch_dt STRING,
    tender_type_cd STRING,
    tot_sales STRING,
    tot_trips STRING,
    tot_pymts STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_control_files_tab_08_redshift} (
    mbrshp_sid STRING,
    tot_sales STRING,
    tot_trips STRING,
    tot_pymts STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_control_files_tab_09_redshift} (
    purch_hdr_id STRING,
    purch_pymt_seq_id STRING,
    mbrshp_nbr STRING,
    purch_dt STRING,
    tender_type_cd STRING,
    cpn_nbr STRING,
    id_mskd_acnt STRING,
    pymt_scanned_or_keyed_ind STRING,
    sales_pymt_amt STRING,
    site_nbr STRING,
    transaction_type_cd STRING,
    register_nbr STRING,
    transaction_nbr STRING,
    sales_channel_id STRING,
    purchase_tm STRING,
    cashier STRING,
    mbr_crd_keyed_vs_scanned_ind STRING,
    tender_id STRING,
    insert_by STRING,
    insert_ts STRING,
    lst_upd_by STRING,
    lst_upd_ts STRING,
    insert_load_nbr STRING,
    lst_upd_load_nbr STRING,
    mbrshp_sid STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_tender_map_group} (
    TENDER_TYPE_CD STRING,
    TENDER_TYPE_CD_DESC STRING,
    GROUPED_TENDER_TYPE STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_bcg_maps_strategic_segments} (
    AGE_RANGE STRING,
    dist_range STRING,
    `25_Percentile_annual_sales` STRING,
    `50_Percentile_annual_sales` STRING,
    `75_Percentile_annual_sales` STRING,
    Avg_annual_sales STRING,
    Count STRING,
    Median_annual_sales STRING,
    Strategic_segment STRING,
    dist_lower STRING,
    dist_upper STRING,
    age_lower STRING,
    age_upper STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_exclusions_brand_exclusions_mixed} (
    CATEGORY_TYPE STRING,
    CATEGORY_CD LONG,
    CATEGORY_DESCRIPTION STRING,
    INCLUDE_OR_EXCLUDE STRING,
    EXCLUSION_TYPE STRING,
    EXCLUSION_SUBTYPE STRING,
    SEASON_MONTH_1 LONG,
    SEASON_MONTH_2 LONG,
    SEASON_MONTH_3 LONG,
    SEASON_MONTH_4 LONG,
    SEASON_MONTH_5 LONG,
    SEASON_MONTH_6 LONG,
    SEASON_MONTH_7 LONG,
    SEASON_MONTH_8 LONG,
    SEASON_MONTH_9 LONG,
    SEASON_MONTH_10 LONG,
    SEASON_MONTH_11 LONG,
    SEASON_MONTH_12 LONG
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

### Silver

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_transaction_detail} (
    PURCH_HDR_ID LONG,
    PURCH_DTL_ID INTEGER,
    PURCH_DT DATE,
    GTIN_CD STRING,
    ARTICLE_NBR STRING,
    MC_CD STRING,
    EXTENDED_PRC_AMT DOUBLE,
    EXTENDED_UNIT_PRC_AMT DOUBLE,
    SALES_QTY DOUBLE,
    SALES_UOM STRING,
    QTY_IN_UNITS INTEGER,
    NORMAL_PRC_AMT DOUBLE,
    NORMAL_UNIT_PRC_AMT DOUBLE,
    REDUCTION_AMT DOUBLE,
    SCANNED_VS_KEYED_IND STRING,
    DISCOUNT_TYPE_CD STRING,
    DISCOUNT_PURCH_DTL_ID INTEGER,
    VOIDED_FLAG STRING,
    VOIDED_PURCH_DTL_ID STRING,
    RSN_CD STRING,
    SALES_CTGRY_CD STRING,
    RETURN_IND STRING,
    REBATE_IND STRING,
    VECTOR_OFFER_ID STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_item} (
    GTIN_CD STRING,
    ARTICLE_DESC STRING,
    ARTICLE_NBR STRING,
    MCH4_CD STRING,
    MCH4_DESC STRING,
    MCH3_CD STRING,
    MCH3_DESC STRING,
    MCH2_CD STRING,
    MCH2_DESC STRING,
    MCH1_CD STRING,
    MCH1_DESC STRING,
    MC_CD STRING,
    MC_DESC STRING,
    AH1_CD STRING,
    AH1_DESC STRING,
    AH2_CD STRING,
    AH2_DESC STRING,
    AH3_CD STRING,
    AH3_DESC STRING,
    AH4_CD STRING,
    AH4_DESC STRING,
    AH5_CD STRING,
    AH5_DESC STRING,
    AH6_CD STRING,
    AH6_DESC STRING,
    BRAND_TYPE STRING,
    EFF_DT DATE,
    EXP_DT DATE,
    REPLACEMENT_ARTICLE STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_brand} (
    ARTICLE_NBR STRING,
    CASE_EXPRESSION STRING,
    BRAND STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_member} (
    MBRSHP_SID LONG,
    MBRSHP_TYPE_ID STRING,
    MBRSHP_FEE_INC DECIMAL(5,2),
    MBRSHP_SUB_TYPE STRING,
    MBRSHP_ENR_DT DATE,
    MBRSHP_EXP_DT DATE,
    MBRSHP_RNWL_DT DATE,
    RWDS_MBR_IND STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_member_history} (
    MBRSHP_SID LONG,
    EFF_DT DATE,
    EXP_DT DATE,
    MBRSHP_STAT_CD STRING,
    MBRSHP_EXP_DT DATE,
    MBRSHP_RNWL_DT DATE,
    MBRSHP_FEE_INC DOUBLE,
    RWDS_MBR_IND STRING,
    RWDS_MBR_ENR_DT DATE,
    CLUB_OF_FREQUENCY INTEGER,
    TEAM_MBR_IND STRING,
    FISCAL_WEEK_END DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_member_extended} (
    MBRSHP_NBR STRING,
    MBRSHP_SID LONG,
    MBRSHP_TYPE_ID INTEGER,
    MBRSHP_FEE_INC DECIMAL(10,3),
    MBRSHP_SUB_TYPE STRING,
    MBRSHP_ENR_DT DATE,
    MBRSHP_EXP_DT DATE,
    MBRSHP_RNWL_DT DATE,
    RWDS_MBR_IND STRING,
    RWDS_MBR_ENR_DT DATE,
    CLUB_OF_FREQUENCY INTEGER,
    MKT_CD STRING,
    AUTO_RNWL_IND STRING,
    ER_SIGNUP_DT DATE,
    HOME_ZIP_CD STRING,
    SIC_CD INTEGER,
    GRP_AFFIL_ID STRING,
    HH_SID INTEGER,
    PRI_SUPP_FHH_IND INTEGER
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_transaction_header} (
    PURCH_HDR_ID LONG,
    MBRSHP_SID LONG,
    SITE_NBR INTEGER,
    PURCH_DT DATE,
    SALES_CHANNEL_ID STRING,
    TOT_SALES_AMT DOUBLE,
    TAX_AMT DOUBLE,
    PURCHASE_TM STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_transaction_payment} (
    MBRSHP_SID LONG,
    PURCH_HDR_ID LONG,
    PURCH_PYMT_SEQ_ID INTEGER,
    TENDER_TYPE_CD STRING,
    CPN_NBR STRING,
    PYMT_SCANNED_OR_KEYED_IND STRING,
    SALES_PYMT_AMT DECIMAL(9,2),
    TENDER_ID STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_fiscal_days} (
    FISCAL_DAY DATE,
    FISCAL_WEEK_START DATE,
    FISCAL_WEEK_END DATE,
    FISCAL_L4W_END DATE,
    FISCAL_L8W_END DATE,
    FISCAL_L12W_END DATE,
    FISCAL_L26W_END DATE,
    FISCAL_L52W_END DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_transaction_fiscal_header} (
    PURCH_HDR_ID LONG,
    MBRSHP_SID LONG,
    SITE_NBR INTEGER,
    PURCH_DT DATE,
    SALES_CHANNEL_ID STRING,
    TOT_SALES_AMT DOUBLE,
    TAX_AMT DOUBLE,
    PURCHASE_TM STRING,
    FISCAL_WEEK_START DATE,
    FISCAL_WEEK_END DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_awards} (
    MBRSHP_SID LONG,
    AWRD_CERT_NBR STRING,
    AWRD_CERT_AMT DOUBLE,
    AWRD_CERT_ISSUE_DT DATE,
    AWRD_CERT_EXP_DT DATE,
    AWRD_CERT_RDMPTN_CD STRING,
    AWRD_PROMO_ID STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_awards_fiscal} (
    MBRSHP_SID LONG,
    AWRD_CERT_NBR STRING,
    AWRD_CERT_AMT DOUBLE,
    AWRD_CERT_ISSUE_DT DATE,
    AWRD_CERT_EXP_DT DATE,
    AWRD_CERT_RDMPTN_CD STRING,
    AWRD_PROMO_ID STRING,
    FISCAL_WEEK_END DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_quotient_id} (
    MBRSHP_SID LONG,
    SIGNUP DATE,
    USERCODE STRING,
    MBRSHP_NBR STRING,
    HAS_QUOTIENT_ID INTEGER
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_transaction_fiscal_payment} (
    PURCH_HDR_ID LONG,
    MBRSHP_SID LONG,
    PURCH_PYMT_SEQ_ID INTEGER,
    TENDER_TYPE_CD STRING,
    CPN_NBR STRING,
    PYMT_SCANNED_OR_KEYED_IND STRING,
    SALES_PYMT_AMT DECIMAL(9,2),
    TENDER_ID STRING,
    PURCH_DT DATE,
    FISCAL_WEEK_START DATE,
    FISCAL_WEEK_END DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_transaction_fiscal_detail} (
    PURCH_HDR_ID LONG,
    PURCH_DTL_ID INTEGER,
    PURCH_DT DATE,
    GTIN_CD STRING,
    ARTICLE_NBR STRING,
    MC_CD STRING,
    EXTENDED_PRC_AMT DOUBLE,
    EXTENDED_UNIT_PRC_AMT DOUBLE,
    SALES_QTY DOUBLE,
    SALES_UOM STRING,
    QTY_IN_UNITS INTEGER,
    NORMAL_PRC_AMT DOUBLE,
    NORMAL_UNIT_PRC_AMT DOUBLE,
    REDUCTION_AMT DOUBLE,
    SCANNED_VS_KEYED_IND STRING,
    DISCOUNT_TYPE_CD STRING,
    DISCOUNT_PURCH_DTL_ID INTEGER,
    VOIDED_FLAG STRING,
    VOIDED_PURCH_DTL_ID STRING,
    RSN_CD STRING,
    SALES_CTGRY_CD STRING,
    RETURN_IND STRING,
    REBATE_IND STRING,
    VECTOR_OFFER_ID STRING,
    MBRSHP_SID LONG,
    SITE_NBR INTEGER,
    FISCAL_WEEK_START DATE,
    FISCAL_WEEK_END DATE,
    ARTICLE_DESC STRING,
    MCH4_CD STRING,
    MCH4_DESC STRING,
    MCH3_CD STRING,
    MCH3_DESC STRING,
    MCH2_CD STRING,
    MCH2_DESC STRING,
    MCH1_CD STRING,
    MCH1_DESC STRING,
    MC_DESC STRING,
    AH1_CD STRING,
    AH1_DESC STRING,
    AH2_CD STRING,
    AH2_DESC STRING,
    AH3_CD STRING,
    AH3_DESC STRING,
    AH4_CD STRING,
    AH4_DESC STRING,
    AH5_CD STRING,
    AH5_DESC STRING,
    AH6_CD STRING,
    AH6_DESC STRING,
    BRAND_TYPE STRING,
    EFF_DT DATE,
    EXP_DT DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_email} (
    MBRSHP_NBR STRING,
    MBRSHP_SID LONG,
    MAIL_ID STRING,
    EMAIL_SUBJECT STRING,
    EMAIL_NAME STRING,
    FIRST_BOUNCE_DATE DATE,
    LAST_BOUNCE_DATE DATE,
    BOUNCE_TOTAL INTEGER,
    FIRST_SEND_DATE DATE,
    LAST_SEND_DATE DATE,
    SEND_TOTAL INTEGER,
    FIRST_OPEN_DATE DATE,
    LAST_OPEN_DATE DATE,
    OPEN_TOTAL INTEGER,
    FIRST_CLICK_DATE DATE,
    LAST_CLICK_DATE DATE,
    CLICK_TOTAL INTEGER,
    FIRST_UNSUB_DATE DATE,
    LAST_UNSUB_DATE DATE,
    UNSUB_TOTAL INTEGER,
    ID LONG
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_email_fiscal} (
    MBRSHP_NBR STRING,
    MBRSHP_SID LONG,
    MAIL_ID STRING,
    EMAIL_SUBJECT STRING,
    EMAIL_NAME STRING,
    FIRST_BOUNCE_DATE DATE,
    LAST_BOUNCE_DATE DATE,
    BOUNCE_TOTAL INTEGER,
    FIRST_SEND_DATE DATE,
    LAST_SEND_DATE DATE,
    SEND_TOTAL INTEGER,
    FIRST_OPEN_DATE DATE,
    LAST_OPEN_DATE DATE,
    OPEN_TOTAL INTEGER,
    FIRST_CLICK_DATE DATE,
    LAST_CLICK_DATE DATE,
    CLICK_TOTAL INTEGER,
    FIRST_UNSUB_DATE DATE,
    LAST_UNSUB_DATE DATE,
    UNSUB_TOTAL INTEGER,
    ID LONG,
    FISCAL_WEEK_START DATE,
    FISCAL_WEEK_END DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_ad_hoc_master_item_with_brand} (
    ARTICLE_NBR STRING,
    GTIN_CD STRING,
    ARTICLE_DESC STRING,
    MCH4_CD STRING,
    MCH4_DESC STRING,
    MCH3_CD STRING,
    MCH3_DESC STRING,
    MCH2_CD STRING,
    MCH2_DESC STRING,
    MCH1_CD STRING,
    MCH1_DESC STRING,
    MC_CD STRING,
    MC_DESC STRING,
    AH1_CD STRING,
    AH1_DESC STRING,
    AH2_CD STRING,
    AH2_DESC STRING,
    AH3_CD STRING,
    AH3_DESC STRING,
    AH4_CD STRING,
    AH4_DESC STRING,
    AH5_CD STRING,
    AH5_DESC STRING,
    AH6_CD STRING,
    AH6_DESC STRING,
    BRAND_TYPE STRING,
    EFF_DT DATE,
    EXP_DT DATE,
    REPLACEMENT_ARTICLE STRING,
    BRAND_DESC STRING,
    BRAND_CD INTEGER
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_census_tract} (
    CENSUS_TRACT STRING,
    ZIP STRING,
    MBRSHP_SID LONG,
    LONGITUDE DOUBLE,
    LATITUDE DOUBLE,
    TRACT_LATITUDE DOUBLE,
    TRACT_LONGITUDE DOUBLE,
    BJS_DISTANCE DOUBLE,
    BJS_DRIVING_DISTANCE DOUBLE,
    BJS_DRIVE_TIME DOUBLE,
    COSTCO_DISTANCE DOUBLE,
    COSTCO_DRIVING_DISTANCE DOUBLE,
    COSTCO_DRIVE_TIME DOUBLE,
    SAMS_DISTANCE DOUBLE,
    SAMS_DRIVING_DISTANCE DOUBLE,
    SAMS_DRIVE_TIME DOUBLE,
    WALMART_DISTANCE DOUBLE,
    WALMART_DRIVING_DISTANCE DOUBLE,
    WALMART_DRIVE_TIME DOUBLE,
    ZIP_DISTANCE DOUBLE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_coupon_clip} (
    MBRSHP_SID LONG,
    MBRSHP_NBR STRING,
    EVENTTYPE STRING,
    EVENTDATETIME DATE,
    OFFERACTIVEDATE DATE,
    OFFERSHUTOFFDATE DATE,
    OFFEREXPIRYDATE DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_coupon_clip_fiscal} (
    MBRSHP_SID LONG,
    MBRSHP_NBR STRING,
    EVENTTYPE STRING,
    EVENTDATETIME DATE,
    OFFERACTIVEDATE DATE,
    OFFERSHUTOFFDATE DATE,
    OFFEREXPIRYDATE DATE,
    FISCAL_WEEK_START DATE,
    FISCAL_WEEK_END DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_skeleton} (
    MBRSHP_SID LONG,
    FISCAL_WEEK_START DATE,
    FISCAL_WEEK_END DATE,
    FISCAL_L4W_END DATE,
    FISCAL_L8W_END DATE,
    FISCAL_L12W_END DATE,
    FISCAL_L26W_END DATE,
    FISCAL_L52W_END DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_club_with_brand} (
    SITE_NBR INTEGER,
    SITE_NAME_2 STRING,
    ADDR_LINE_2 STRING,
    CITY_NAME STRING,
    STATE_CD STRING,
    ZIP_CD STRING,
    ZN_NBR INTEGER,
    RGN_NBR INTEGER,
    SITE_TYPE STRING,
    COMP_STTS STRING,
    FIRST_FW_HAS_GAS DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_club_square_with_brand} (
    SITE_NBR INTEGER,
    CATEGORY STRING,
    CATEGORY_LVL STRING,
    SALES DOUBLE,
    HAS_CATEGORY INTEGER
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_master_member_basic} (
    mbr_sid STRING,
    mbr_prmry_sid STRING,
    hh_rollup_prmry_sid STRING,
    mbr_stts_cd STRING,
    mbr_mkt_cd STRING,
    mbr_bsn_sic_cd STRING,
    mbr_bsn_sic_frst4_cd STRING,
    mbr_bsn_sic_lst4_cd STRING,
    mbr_grp_affilatn_cd STRING,
    mbr_rwd_typ STRING,
    mbr_clb_of_mbrshp STRING,
    mbr_typ_cd STRING,
    mbr_clb_usr_lst_updt_id STRING,
    mbr_ic_spplmntl_cnt STRING,
    mbr_upgrd_prmpt_cnt STRING,
    mbr_fuel_prmtn_elgbl_ind STRING,
    mbr_chk_stts_cd STRING,
    mbr_chk_fscl_ytd_cnt STRING,
    mbr_lst_updt_dt STRING,
    mbr_lst_updt_clb_nbr STRING,
    mbr_lst_sls_txn_dt STRING,
    mbr_enrl_dt STRING,
    mbr_exp_dt STRING,
    mbr_prev_exp_dt STRING,
    mbr_prorate_mth STRING,
    mbr_rnwl_dt STRING,
    mbr_mfi_amt STRING,
    mbr_rnwl_clb_nbr STRING,
    mbr_aqustn_prmtn_cd STRING,
    mbr_prescrn_id STRING,
    mbr_lst_prescrn_dt STRING,
    mbr_prescrn_exp_dt STRING,
    mbr_prescrn_src_ind STRING,
    mbr_pos_upgrd_prmpt_dt STRING,
    mbr_ezr_ask_dt STRING,
    mbr_ezr_signup_dt STRING,
    mbr_ezr_optout_dt STRING,
    mbr_addr_updt_dt STRING,
    mbr_addr_updt_channel_cd STRING,
    mbr_email_valid_ind STRING,
    mbr_email_frqncy_ind STRING,
    mbr_email_enrl_via_net STRING,
    mbr_email_rnwl_via_net STRING,
    mbr_email_upgrd_via_net STRING,
    mbr_email_addr_updt_channel_cd STRING,
    mbr_email_addr_updt_clb STRING,
    mbr_email_addr_updt_dt STRING,
    mbr_email_optin_prefr_updt_dt STRING,
    mbr_email_optin_prefr_updt_clb STRING,
    mbr_email_optin_prefr_updt_src STRING,
    mbr_dlvry_addr_ind STRING,
    mbr_tele_mrktng_optin_ind STRING,
    mbr_prmtn_optin_ind STRING,
    mbr_ezr_stts_cd STRING,
    mbr_ezr_stts_ind STRING,
    mbr_profile_ind STRING,
    mbr_household_ind STRING,
    mbr_tmbr_ind STRING,
    trial_mbr_ind STRING,
    mbr_do_not_offer_email_ind STRING,
    mbr_cell_txt_msg_opt_ind STRING,
    mbr_cell_txt_opt_chg_dt STRING,
    mbr_cell_txt_opt_chg_clb STRING,
    mbr_cell_txt_opt_chg_src_cd STRING,
    mbr_household_cnvrtd_ind STRING,
    mbr_free_sup_ind STRING,
    mbr_awrd_shrd_ind STRING,
    mbr_awrd_shrd_chg_dt STRING,
    mbr_scan_go_elgbl_ind STRING,
    mbr_phone_typ_cd STRING,
    mbr_household_cnvrtd_prnt_dt STRING,
    mbr_prtnr_sid STRING,
    mbr_arinfo_flg STRING,
    mbr_prmry_flg STRING,
    mbr_online_only_flg STRING,
    mbr_rwd_enrl_dt STRING,
    mbr_rwd_downgrade_to_basic_dt STRING,
    mbr_rwd_stts_change_dt STRING,
    mbr_rwd_change_to_stts_cd STRING,
    mbr_rwd_cobrand_enroll_dt STRING,
    ads_acct_stts_ind STRING,
    ads_acct_open_dt STRING,
    ads_acct_close_dt STRING,
    drvd_clb_of_frqncy STRING,
    drvd_clb_of_frqncy_rollup STRING,
    drvd_mktble_univ_cd STRING,
    drvd_mbr_sub_type_cd STRING,
    drvd_tenure_mth STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_bcg_maps_ah5_customer_facing_desc}  (
    AH5_CD STRING,
    AH5_DESC STRING,
    MBR_FACING_CATEGORY STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_transaction_fiscal_detail_isnr} (
    PURCH_HDR_ID LONG,
    PURCH_DTL_ID INTEGER,
    PURCH_DT DATE,
    GTIN_CD STRING,
    ARTICLE_NBR STRING,
    MC_CD STRING,
    EXTENDED_PRC_AMT DOUBLE,
    EXTENDED_UNIT_PRC_AMT DOUBLE,
    SALES_QTY DOUBLE,
    SALES_UOM STRING,
    QTY_IN_UNITS INTEGER,
    NORMAL_PRC_AMT DOUBLE,
    NORMAL_UNIT_PRC_AMT DOUBLE,
    REDUCTION_AMT DOUBLE,
    SCANNED_VS_KEYED_IND STRING,
    DISCOUNT_TYPE_CD STRING,
    DISCOUNT_PURCH_DTL_ID INTEGER,
    VOIDED_FLAG STRING,
    VOIDED_PURCH_DTL_ID STRING,
    RSN_CD STRING,
    SALES_CTGRY_CD STRING,
    RETURN_IND STRING,
    REBATE_IND STRING,
    VECTOR_OFFER_ID STRING,
    MBRSHP_SID LONG,
    SITE_NBR INTEGER,
    FISCAL_WEEK_START DATE,
    FISCAL_WEEK_END DATE,
    ARTICLE_DESC STRING,
    MCH4_CD STRING,
    MCH4_DESC STRING,
    MCH3_CD STRING,
    MCH3_DESC STRING,
    MCH2_CD STRING,
    MCH2_DESC STRING,
    MCH1_CD STRING,
    MCH1_DESC STRING,
    MC_DESC STRING,
    AH1_CD STRING,
    AH1_DESC STRING,
    AH2_CD STRING,
    AH2_DESC STRING,
    AH3_CD STRING,
    AH3_DESC STRING,
    AH4_CD STRING,
    AH4_DESC STRING,
    AH5_CD STRING,
    AH5_DESC STRING,
    AH6_CD STRING,
    AH6_DESC STRING,
    BRAND_TYPE STRING,
    EFF_DT DATE,
    EXP_DT DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_transaction_fiscal_detail_gas_nr} (
    PURCH_HDR_ID LONG,
    PURCH_DTL_ID INTEGER,
    PURCH_DT DATE,
    GTIN_CD STRING,
    ARTICLE_NBR STRING,
    MC_CD STRING,
    EXTENDED_PRC_AMT DOUBLE,
    EXTENDED_UNIT_PRC_AMT DOUBLE,
    SALES_QTY DOUBLE,
    SALES_UOM STRING,
    QTY_IN_UNITS INTEGER,
    NORMAL_PRC_AMT DOUBLE,
    NORMAL_UNIT_PRC_AMT DOUBLE,
    REDUCTION_AMT DOUBLE,
    SCANNED_VS_KEYED_IND STRING,
    DISCOUNT_TYPE_CD STRING,
    DISCOUNT_PURCH_DTL_ID INTEGER,
    VOIDED_FLAG STRING,
    VOIDED_PURCH_DTL_ID STRING,
    RSN_CD STRING,
    SALES_CTGRY_CD STRING,
    RETURN_IND STRING,
    REBATE_IND STRING,
    VECTOR_OFFER_ID STRING,
    MBRSHP_SID LONG,
    SITE_NBR INTEGER,
    FISCAL_WEEK_START DATE,
    FISCAL_WEEK_END DATE,
    ARTICLE_DESC STRING,
    MCH4_CD STRING,
    MCH4_DESC STRING,
    MCH3_CD STRING,
    MCH3_DESC STRING,
    MCH2_CD STRING,
    MCH2_DESC STRING,
    MCH1_CD STRING,
    MCH1_DESC STRING,
    MC_DESC STRING,
    AH1_CD STRING,
    AH1_DESC STRING,
    AH2_CD STRING,
    AH2_DESC STRING,
    AH3_CD STRING,
    AH3_DESC STRING,
    AH4_CD STRING,
    AH4_DESC STRING,
    AH5_CD STRING,
    AH5_DESC STRING,
    AH6_CD STRING,
    AH6_DESC STRING,
    BRAND_TYPE STRING,
    EFF_DT DATE,
    EXP_DT DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_bcg_maps_tender_type_group_csv} (
    TENDER_TYPE_CD STRING,
    TENDER_TYPE_CD_DESC STRING,
    GROUPED_TENDER_TYPE STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_bcg_maps_strategic_segments} (
    AGE_RANGE STRING,
    dist_range STRING,
    `25_Percentile_annual_sales` DOUBLE,
    `50_Percentile_annual_sales` DOUBLE,
    `75_Percentile_annual_sales` DOUBLE,
    Avg_annual_sales DOUBLE,
    Count LONG,
    Median_annual_sales DOUBLE,
    Strategic_segment STRING,
    dist_lower DOUBLE,
    dist_upper DOUBLE,
    age_lower DOUBLE,
    age_upper DOUBLE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_exclusions_brand_exclusions_mixed} (
    CATEGORY_TYPE STRING,
    CATEGORY_CD LONG,
    CATEGORY_DESCRIPTION STRING,
    INCLUDE_OR_EXCLUDE STRING,
    EXCLUSION_TYPE STRING,
    EXCLUSION_SUBTYPE STRING,
    SEASON_MONTH_1 LONG,
    SEASON_MONTH_2 LONG,
    SEASON_MONTH_3 LONG,
    SEASON_MONTH_4 LONG,
    SEASON_MONTH_5 LONG,
    SEASON_MONTH_6 LONG,
    SEASON_MONTH_7 LONG,
    SEASON_MONTH_8 LONG,
    SEASON_MONTH_9 LONG,
    SEASON_MONTH_10 LONG,
    SEASON_MONTH_11 LONG,
    SEASON_MONTH_12 LONG
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

#### Generation of Silver archive tables

In [0]:
from pyspark.sql import functions as F

CATALOG = catalog_name
SCHEMA  = silver_schema_name
FULL_SCHEMA = f"{CATALOG}.{SCHEMA}"

tables_with_archive = [t.name.replace('_archive', '') for t in spark.catalog.listTables(f"{catalog_name}.{silver_schema_name}") if t.tableType == "MANAGED" and "_archive" in t.name]

tables = [t.name for t in spark.catalog.listTables(f"{catalog_name}.pe_slv") if t.tableType == "MANAGED" and t.name not in tables_with_archive and '_archive' not in t.name and '_sampled' not in t.name]

results = []
for tbl in tables:
    if '_comparison' in tbl or tbl in ['master_member_basic', 'bcg_maps_ah5_customer_facing_desc', 'bcg_maps_strategic_segments', 'bcg_maps_tender_type_group_csv', 'exclusions_brand_exclusions_mixed']:
        continue
    src = f"{FULL_SCHEMA}.`{tbl}`"
    dst = f"{FULL_SCHEMA}.`{tbl}_archive`"
    try:
        spark.sql(f"CREATE TABLE IF NOT EXISTS {dst} LIKE {src}")
        spark.sql(f"ALTER TABLE {dst} ADD COLUMNS (run_date DATE)")
        spark.sql(f"ALTER TABLE {dst} ALTER COLUMN run_date SET NOT NULL")
        results.append((tbl, "created/updated"))
    except Exception as e:
        results.append((tbl, f"error: {str(e)}"))

for src, dst in intermediate_sampled_tables:
    try:
        spark.sql(f"CREATE TABLE IF NOT EXISTS {dst} LIKE {src}")
        results.append((dst, "created/updated"))
    except Exception as e:
        results.append((dst, f"error: {str(e)}"))

### DNA

In [0]:
schema = StructType([
    StructField("MBRSHP_SID", LongType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True),
    StructField("L52W_G4W_STDEV_TRIPS", DoubleType(), True),
    StructField("L52W_G4W_STDEV_SPEND", DoubleType(), True),
    StructField("WEEK_TRIPS", LongType(), True),
    StructField("LAST_FOUR_WEEK_TRIPS", LongType(), True),
    StructField("LAST_EIGHT_WEEK_TRIPS", LongType(), True),
    StructField("LAST_TWELVE_WEEK_TRIPS", LongType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_TRIPS", LongType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_TRIPS", LongType(), True),
    StructField("WEEK_SPEND", DoubleType(), True),
    StructField("FW_SPEND_IN_STORE", DoubleType(), True),
    StructField("WEEK_UNITS", LongType(), True),
    StructField("WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("FW_GAS_TRIPS", LongType(), True),
    StructField("FW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("FW_GAS_SPEND", DoubleType(), True),
    StructField("FW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("FW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("FW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("LAST_FOUR_WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("LAST_EIGHT_WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("LAST_TWELVE_WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("WEEK_TRANSACTIONS", LongType(), True)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_cubes_transaction_1,
    primary_keys=["MBRSHP_SID", "FISCAL_WEEK_END"],
    schema=schema,
    description="Member DNA - Transaction 1"
)

In [0]:
schema = StructType([
    StructField("MBRSHP_SID", LongType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True),
    StructField("LAST_FOUR_WEEK_SPEND", DoubleType(), True),
    StructField("LAST_EIGHT_WEEK_SPEND", DoubleType(), True),
    StructField("LAST_TWELVE_WEEK_SPEND", DoubleType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_SPEND", DoubleType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_SPEND", DoubleType(), True),
    StructField("LFOURW_SPEND_IN_STORE", DoubleType(), True),
    StructField("LEIGHTW_SPEND_IN_STORE", DoubleType(), True),
    StructField("LTWELVEW_SPEND_IN_STORE", DoubleType(), True),
    StructField("LTWENTY-SIXW_SPEND_IN_STORE", DoubleType(), True),
    StructField("LFIFTY-TWOW_SPEND_IN_STORE", DoubleType(), True),
    StructField("LAST_FOUR_WEEK_UNITS", LongType(), True),
    StructField("LAST_EIGHT_WEEK_UNITS", LongType(), True),
    StructField("LAST_TWELVE_WEEK_UNITS", LongType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_UNITS", LongType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_UNITS", LongType(), True),
    StructField("LAST_FOUR_WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("LAST_EIGHT_WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("LAST_TWELVE_WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("L_FOURW_GAS_TRIPS", LongType(), True),
    StructField("L_EIGHTW_GAS_TRIPS", LongType(), True),
    StructField("L_TWELVEW_GAS_TRIPS", LongType(), True),
    StructField("L_TWENTY-SIXW_GAS_TRIPS", LongType(), True),
    StructField("L_FIFTY-TWOW_GAS_TRIPS", LongType(), True),
    StructField("L_FOURW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("L_EIGHTW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("L_TWELVEW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("L_TWENTY-SIXW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("L_FIFTY-TWOW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("L_FOURW_GAS_SPEND", DoubleType(), True),
    StructField("L_EIGHTW_GAS_SPEND", DoubleType(), True),
    StructField("L_TWELVEW_GAS_SPEND", DoubleType(), True),
    StructField("L_TWENTY-SIXW_GAS_SPEND", DoubleType(), True),
    StructField("L_FIFTY-TWOW_GAS_SPEND", DoubleType(), True),
    StructField("L_FOURW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("L_EIGHTW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("L_TWELVEW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("L_TWENTY-SIXW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("L_FIFTY-TWOW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("L_FOURW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("L_EIGHTW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("L_TWELVEW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("L_TWENTY-SIXW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("L_FIFTY-TWOW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("L_FOURW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("L_EIGHTW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("L_TWELVEW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("L_TWENTY-SIXW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("L_FIFTY-TWOW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("LAST_FOUR_WEEK_TRANSACTIONS", LongType(), True),
    StructField("LAST_EIGHT_WEEK_TRANSACTIONS", LongType(), True),
    StructField("LAST_TWELVE_WEEK_TRANSACTIONS", LongType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_TRANSACTIONS", LongType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_TRANSACTIONS", LongType(), True)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_cubes_transaction_2,
    primary_keys=["MBRSHP_SID", "FISCAL_WEEK_END"],
    schema=schema,
    description="Member DNA - Transaction 2"
)

In [0]:
schema = StructType([
    StructField("MBRSHP_SID", LongType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True),
    StructField("L52W_CREDIT", DecimalType(38, 2), False),
    StructField("L52W_DEBIT", DecimalType(38, 2), False),
    StructField("L52W_CASH", DecimalType(38, 2), False),
    StructField("L52W_COUPON", DecimalType(38, 2), False),
    StructField("L52W_EBT", DecimalType(38, 2), False),
    StructField("L52W_BJS", DecimalType(38, 2), False),
    StructField("FW_HAS_BOUGHT_MEN", IntegerType(), False),
    StructField("L52W_HAS_BOUGHT_MEN", IntegerType(), False),
    StructField("FW_HAS_BOUGHT_WOMEN", IntegerType(), False),
    StructField("L52W_HAS_BOUGHT_WOMEN", IntegerType(), False),
    StructField("FW_HAS_BOUGHT_PET", IntegerType(), False),
    StructField("L52W_HAS_BOUGHT_PET", IntegerType(), False),
    StructField("FW_HAS_BOUGHT_CHILDREN", IntegerType(), False),
    StructField("L52W_HAS_BOUGHT_CHILDREN", IntegerType(), False),
    StructField("FW_HAS_BOUGHT_BABY", IntegerType(), False),
    StructField("L52W_HAS_BOUGHT_BABY", IntegerType(), False),
    StructField("L52W_MEDIAN_BASKETSIZE", FloatType(), False),
    StructField("L52W_DISTINCT_CATEGORIES", LongType(), False),
    StructField("LAST_FISCAL_WEEK_TRIP", DateType(), True),
    StructField("LAST_TRIP", DateType(), True),
    StructField("DAYS_SINCE_LAST_TRIP", IntegerType(), True),
    StructField("L52W_MAX_INTERVAL", IntegerType(), True),
    StructField("L52W_MIN_INTERVAL", IntegerType(), True)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_cubes_transaction_3,
    primary_keys=["MBRSHP_SID", "FISCAL_WEEK_END"],
    schema=schema,
    description="Member DNA - Transaction 3"
)

In [0]:
schema = StructType([
    StructField("LATEST_MBRSHP_NBR", StringType(), True),
    StructField("ZIP", StringType(), True),
    StructField("MBRSHP_SID", LongType(), True),
    StructField("EFF_DT", DateType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True),
    StructField("LATEST_MBRSHP_TYPE_ID", StringType(), True),
    StructField("LATEST_MBRSHP_FEE_INC", DecimalType(5, 2), True),
    StructField("LATEST_MBRSHP_SUB_TYPE", StringType(), True),
    StructField("LATEST_MBRSHP_ENR_DT", DateType(), True),
    StructField("LATEST_MBRSHP_EXP_DT", DateType(), True),
    StructField("LATEST_MBRSHP_RNWL_DT", DateType(), True),
    StructField("LATEST_RWDS_MBR_IND", StringType(), True),
    StructField("LATEST_MKT_CD", StringType(), True),
    StructField("LATEST_HOME_ZIP_CD", StringType(), True),
    StructField("LATEST_SIC_CD", IntegerType(), True),
    StructField("LATEST_CLUB_OF_FREQUENCY", IntegerType(), True),
    StructField("LATEST_PRI_SUPP_FHH_IND", IntegerType(), True),
    StructField("LATEST_GRP_AFFIL_ID", StringType(), True),
    StructField("LATEST_ER_SIGNUP_DT", DateType(), True),
    StructField("LATEST_AUTO_RNWL_IND", StringType(), True),
    StructField("LATEST_TM_MBR_IND", IntegerType(), True),
    StructField("LATEST_TRIAL_MBR_IND", IntegerType(), True),
    StructField("LATEST_MFI_TIER", IntegerType(), True),
    StructField("EXP_DT", DateType(), True),
    StructField("MBRSHP_STAT_CD", StringType(), True),
    StructField("MBRSHP_EXP_DT", DateType(), True),
    StructField("MBRSHP_RNWL_DT", DateType(), True),
    StructField("MBRSHP_FEE_INC", DoubleType(), True),
    StructField("RWDS_MBR_IND", StringType(), True),
    StructField("RWDS_MBR_ENR_DT", DateType(), True),
    StructField("CLUB_OF_FREQUENCY", IntegerType(), True),
    StructField("TEAM_MBR_IND", StringType(), True),
    StructField("FIRST_MBRSHP_FEE_INC", DoubleType(), True),
    StructField("ZIP_DISTANCE", DoubleType(), True),
    StructField("BJS_DRIVING_DISTANCE", DoubleType(), True),
    StructField("BJS_DISTANCE", DoubleType(), True),
    StructField("BJS_DRIVE_TIME", DoubleType(), True),
    StructField("WALMART_DRIVE_TIME", DoubleType(), True),
    StructField("WALMART_DRIVING_DISTANCE", DoubleType(), True),
    StructField("WALMART_DISTANCE", DoubleType(), True),
    StructField("COSTCO_DRIVE_TIME", DoubleType(), True),
    StructField("COSTCO_DRIVING_DISTANCE", DoubleType(), True),
    StructField("COSTCO_DISTANCE", DoubleType(), True),
    StructField("SAMS_DRIVE_TIME", DoubleType(), True),
    StructField("SAMS_DRIVING_DISTANCE", DoubleType(), True),
    StructField("SAMS_DISTANCE", DoubleType(), True),
    StructField("TENURE", IntegerType(), True),
    StructField("TENURE_GROUP", StringType(), True),
    StructField("DAYS_UNTIL_EXP", IntegerType(), True),
    StructField("DAYS_SINCE_LAST_RNWL", IntegerType(), True),
    StructField("NUM_OF_RNWLS", IntegerType(), True),
    StructField("HAS_QUOTIENT_ID", IntegerType(), True)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_cubes_member,
    primary_keys=["MBRSHP_SID", "FISCAL_WEEK_END"],
    schema=schema,
    description="Member DNA - Member features"
)

In [0]:
schema = StructType([
    StructField("L52W_PREFERRED_CLUB_NBR", IntegerType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True),
    StructField("MBRSHP_SID", LongType(), True),
    StructField("L52W_PREFERRED_CLUB_TRIPS", LongType(), True),
    StructField("L52W_PERCENT_TRIPS_PREFERRED_CLUB", DoubleType(), True),
    StructField("DUMMY_MBR", IntegerType(), True),
    StructField("L12W_SPEND_OVER_P12W_SPEND", DoubleType(), True),
    StructField("L26W_SPEND_OVER_P26W_SPEND", DoubleType(), True),
    StructField("L12W_TRIPS_OVER_P12W_TRIPS", DoubleType(), True),
    StructField("L26W_TRIPS_OVER_P26W_TRIPS", DoubleType(), True),
    StructField("STRATEGIC_MBR_HEADROOM", DoubleType(), True),
    StructField("DIST_RANGE", StringType(), True),
    StructField("AGE_RANGE", StringType(), True),
    StructField("IS_STRATEGIC_MBR", IntegerType(), True),
    StructField("PREFERRED_CLUB_HAS_GAS", IntegerType(), True)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_cubes_misc,
    primary_keys=["MBRSHP_SID", "FISCAL_WEEK_END"],
    schema=schema,
    description="Member DNA - Misc features"
)

In [0]:
schema = StructType([
    StructField("MBRSHP_SID", LongType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True),
    StructField("FISCAL_WEEK_START", DateType(), True),
    StructField("FISCAL_L4W_END", DateType(), True),
    StructField("FISCAL_L8W_END", DateType(), True),
    StructField("FISCAL_L12W_END", DateType(), True),
    StructField("FISCAL_L26W_END", DateType(), True),
    StructField("FISCAL_L52W_END", DateType(), True),
    StructField("L52W_G4W_STDEV_TRIPS", DoubleType(), True),
    StructField("L52W_G4W_STDEV_SPEND", DoubleType(), True),
    StructField("WEEK_TRIPS", LongType(), True),
    StructField("LAST_FOUR_WEEK_TRIPS", LongType(), True),
    StructField("LAST_EIGHT_WEEK_TRIPS", LongType(), True),
    StructField("LAST_TWELVE_WEEK_TRIPS", LongType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_TRIPS", LongType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_TRIPS", LongType(), True),
    StructField("WEEK_SPEND", DoubleType(), True),
    StructField("FW_SPEND_IN_STORE", DoubleType(), True),
    StructField("WEEK_UNITS", LongType(), True),
    StructField("WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("FW_GAS_TRIPS", LongType(), True),
    StructField("FW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("FW_GAS_SPEND", DoubleType(), True),
    StructField("FW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("FW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("FW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("LAST_FOUR_WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("LAST_EIGHT_WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("LAST_TWELVE_WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_DISTINCT_DAYS", LongType(), True),
    StructField("WEEK_TRANSACTIONS", LongType(), True),
    StructField("LAST_FOUR_WEEK_SPEND", DoubleType(), True),
    StructField("LAST_EIGHT_WEEK_SPEND", DoubleType(), True),
    StructField("LAST_TWELVE_WEEK_SPEND", DoubleType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_SPEND", DoubleType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_SPEND", DoubleType(), True),
    StructField("LFOURW_SPEND_IN_STORE", DoubleType(), True),
    StructField("LEIGHTW_SPEND_IN_STORE", DoubleType(), True),
    StructField("LTWELVEW_SPEND_IN_STORE", DoubleType(), True),
    StructField("LTWENTY-SIXW_SPEND_IN_STORE", DoubleType(), True),
    StructField("LFIFTY-TWOW_SPEND_IN_STORE", DoubleType(), True),
    StructField("LAST_FOUR_WEEK_UNITS", LongType(), True),
    StructField("LAST_EIGHT_WEEK_UNITS", LongType(), True),
    StructField("LAST_TWELVE_WEEK_UNITS", LongType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_UNITS", LongType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_UNITS", LongType(), True),
    StructField("LAST_FOUR_WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("LAST_EIGHT_WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("LAST_TWELVE_WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_UNITS_OVER_FIFTY", LongType(), True),
    StructField("L_FOURW_GAS_TRIPS", LongType(), True),
    StructField("L_EIGHTW_GAS_TRIPS", LongType(), True),
    StructField("L_TWELVEW_GAS_TRIPS", LongType(), True),
    StructField("L_TWENTY-SIXW_GAS_TRIPS", LongType(), True),
    StructField("L_FIFTY-TWOW_GAS_TRIPS", LongType(), True),
    StructField("L_FOURW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("L_EIGHTW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("L_TWELVEW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("L_TWENTY-SIXW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("L_FIFTY-TWOW_GAS_DISTINCT_DAYS", LongType(), True),
    StructField("L_FOURW_GAS_SPEND", DoubleType(), True),
    StructField("L_EIGHTW_GAS_SPEND", DoubleType(), True),
    StructField("L_TWELVEW_GAS_SPEND", DoubleType(), True),
    StructField("L_TWENTY-SIXW_GAS_SPEND", DoubleType(), True),
    StructField("L_FIFTY-TWOW_GAS_SPEND", DoubleType(), True),
    StructField("L_FOURW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("L_EIGHTW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("L_TWELVEW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("L_TWENTY-SIXW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("L_FIFTY-TWOW_GAS_AND_STORE_DISTINCT_DAYS", LongType(), True),
    StructField("L_FOURW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("L_EIGHTW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("L_TWELVEW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("L_TWENTY-SIXW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("L_FIFTY-TWOW_ECOMMERCE_SPEND", DoubleType(), True),
    StructField("L_FOURW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("L_EIGHTW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("L_TWELVEW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("L_TWENTY-SIXW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("L_FIFTY-TWOW_ECOMMERCE_TRIPS", LongType(), True),
    StructField("LAST_FOUR_WEEK_TRANSACTIONS", LongType(), True),
    StructField("LAST_EIGHT_WEEK_TRANSACTIONS", LongType(), True),
    StructField("LAST_TWELVE_WEEK_TRANSACTIONS", LongType(), True),
    StructField("LAST_TWENTY-SIX_WEEK_TRANSACTIONS", LongType(), True),
    StructField("LAST_FIFTY-TWO_WEEK_TRANSACTIONS", LongType(), True),
    StructField("L52W_CREDIT", DecimalType(38, 2), True),
    StructField("L52W_DEBIT", DecimalType(38, 2), True),
    StructField("L52W_CASH", DecimalType(38, 2), True),
    StructField("L52W_COUPON", DecimalType(38, 2), True),
    StructField("L52W_EBT", DecimalType(38, 2), True),
    StructField("L52W_BJS", DecimalType(38, 2), True),
    StructField("FW_HAS_BOUGHT_MEN", IntegerType(), True),
    StructField("L52W_HAS_BOUGHT_MEN", IntegerType(), True),
    StructField("FW_HAS_BOUGHT_WOMEN", IntegerType(), True),
    StructField("L52W_HAS_BOUGHT_WOMEN", IntegerType(), True),
    StructField("FW_HAS_BOUGHT_PET", IntegerType(), True),
    StructField("L52W_HAS_BOUGHT_PET", IntegerType(), True),
    StructField("FW_HAS_BOUGHT_CHILDREN", IntegerType(), True),
    StructField("L52W_HAS_BOUGHT_CHILDREN", IntegerType(), True),
    StructField("FW_HAS_BOUGHT_BABY", IntegerType(), True),
    StructField("L52W_HAS_BOUGHT_BABY", IntegerType(), True),
    StructField("L52W_MEDIAN_BASKETSIZE", FloatType(), True),
    StructField("L52W_DISTINCT_CATEGORIES", LongType(), True),
    StructField("LAST_FISCAL_WEEK_TRIP", DateType(), True),
    StructField("LAST_TRIP", DateType(), True),
    StructField("DAYS_SINCE_LAST_TRIP", IntegerType(), True),
    StructField("L52W_MAX_INTERVAL", IntegerType(), True),
    StructField("L52W_MIN_INTERVAL", IntegerType(), True),
    StructField("FW_ATC_CPN_RED", LongType(), True),
    StructField("FW_ATC_CPN_SPEND", DoubleType(), True),
    StructField("L4W_ATC_CPN_RED", LongType(), True),
    StructField("L4W_ATC_CPN_SPEND", DoubleType(), True),
    StructField("L8W_ATC_CPN_RED", LongType(), True),
    StructField("L8W_ATC_CPN_SPEND", DoubleType(), True),
    StructField("L12W_ATC_CPN_RED", LongType(), True),
    StructField("L12W_ATC_CPN_SPEND", DoubleType(), True),
    StructField("L26W_ATC_CPN_RED", LongType(), True),
    StructField("L26W_ATC_CPN_SPEND", DoubleType(), True),
    StructField("L52W_ATC_CPN_RED", LongType(), True),
    StructField("L52W_ATC_CPN_SPEND", DoubleType(), True),
    StructField("FW_CPN_CLP_COUNT", LongType(), True),
    StructField("L4W_CPN_CLP_COUNT", LongType(), True),
    StructField("L8W_CPN_CLP_COUNT", LongType(), True),
    StructField("L12W_CPN_CLP_COUNT", LongType(), True),
    StructField("L26W_CPN_CLP_COUNT", LongType(), True),
    StructField("L52W_CPN_CLP_COUNT", LongType(), True),
    StructField("FW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("FW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("FW_COUPON_SAVINGS", DoubleType(), True),
    StructField("FW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LAST_FISCAL_WEEK_COUPON_REDEEMED", DateType(), True),
    StructField("LAST_COUPON_REDEEMED", DateType(), True),
    StructField("DAYS_SINCE_LAST_COUPON_REDEEMED", IntegerType(), True),
    StructField("LAST_FISCAL_WEEK_EMAIL_OPENED", DateType(), True),
    StructField("LAST_EMAIL_OPENED", DateType(), True),
    StructField("DAYS_SINCE_LAST_EMAIL_OPENED", IntegerType(), True),
    StructField("LAST_FISCAL_WEEK_ATC_CLIPPED", DateType(), True),
    StructField("LAST_ATC_CLIPPED", DateType(), True),
    StructField("DAYS_SINCE_LAST_ATC_CLIPPED", IntegerType(), True),
    StructField("EMAIL_OPEN_RATE", DoubleType(), True),
    StructField("FW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("cpn_channel", StringType(), True),
    StructField("LFOURW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LEIGHTW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LTWELVEW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LTWENTY-SIXW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LFIFTY-TWOW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LFOURW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("LFOURW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("LFOURW_COUPON_SAVINGS", DoubleType(), True),
    StructField("LFOURW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LEIGHTW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("LEIGHTW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("LEIGHTW_COUPON_SAVINGS", DoubleType(), True),
    StructField("LEIGHTW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LTWELVEW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("LTWELVEW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("LTWELVEW_COUPON_SAVINGS", DoubleType(), True),
    StructField("LTWELVEW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LTWENTY-SIXW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("LTWENTY-SIXW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("LTWENTY-SIXW_COUPON_SAVINGS", DoubleType(), True),
    StructField("LTWENTY-SIXW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LFIFTY-TWOW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("LFIFTY-TWOW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("LFIFTY-TWOW_COUPON_SAVINGS", DoubleType(), True),
    StructField("LFIFTY-TWOW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LATEST_MBRSHP_NBR", StringType(), True),
    StructField("ZIP", StringType(), True),
    StructField("EFF_DT", DateType(), True),
    StructField("LATEST_MBRSHP_TYPE_ID", StringType(), True),
    StructField("LATEST_MBRSHP_FEE_INC", DecimalType(5, 2), True),
    StructField("LATEST_MBRSHP_SUB_TYPE", StringType(), True),
    StructField("LATEST_MBRSHP_ENR_DT", DateType(), True),
    StructField("LATEST_MBRSHP_EXP_DT", DateType(), True),
    StructField("LATEST_MBRSHP_RNWL_DT", DateType(), True),
    StructField("LATEST_RWDS_MBR_IND", StringType(), True),
    StructField("LATEST_MKT_CD", StringType(), True),
    StructField("LATEST_HOME_ZIP_CD", StringType(), True),
    StructField("LATEST_SIC_CD", IntegerType(), True),
    StructField("LATEST_CLUB_OF_FREQUENCY", IntegerType(), True),
    StructField("LATEST_PRI_SUPP_FHH_IND", IntegerType(), True),
    StructField("LATEST_GRP_AFFIL_ID", StringType(), True),
    StructField("LATEST_ER_SIGNUP_DT", DateType(), True),
    StructField("LATEST_AUTO_RNWL_IND", StringType(), True),
    StructField("LATEST_TM_MBR_IND", IntegerType(), True),
    StructField("LATEST_TRIAL_MBR_IND", IntegerType(), True),
    StructField("LATEST_MFI_TIER", IntegerType(), True),
    StructField("EXP_DT", DateType(), True),
    StructField("MBRSHP_STAT_CD", StringType(), True),
    StructField("MBRSHP_EXP_DT", DateType(), True),
    StructField("MBRSHP_RNWL_DT", DateType(), True),
    StructField("MBRSHP_FEE_INC", DoubleType(), True),
    StructField("RWDS_MBR_IND", StringType(), True),
    StructField("RWDS_MBR_ENR_DT", DateType(), True),
    StructField("CLUB_OF_FREQUENCY", IntegerType(), True),
    StructField("TEAM_MBR_IND", StringType(), True),
    StructField("FIRST_MBRSHP_FEE_INC", DoubleType(), True),
    StructField("ZIP_DISTANCE", DoubleType(), True),
    StructField("BJS_DRIVING_DISTANCE", DoubleType(), True),
    StructField("BJS_DISTANCE", DoubleType(), True),
    StructField("BJS_DRIVE_TIME", DoubleType(), True),
    StructField("WALMART_DRIVE_TIME", DoubleType(), True),
    StructField("WALMART_DRIVING_DISTANCE", DoubleType(), True),
    StructField("WALMART_DISTANCE", DoubleType(), True),
    StructField("COSTCO_DRIVE_TIME", DoubleType(), True),
    StructField("COSTCO_DRIVING_DISTANCE", DoubleType(), True),
    StructField("COSTCO_DISTANCE", DoubleType(), True),
    StructField("SAMS_DRIVE_TIME", DoubleType(), True),
    StructField("SAMS_DRIVING_DISTANCE", DoubleType(), True),
    StructField("SAMS_DISTANCE", DoubleType(), True),
    StructField("TENURE", IntegerType(), True),
    StructField("TENURE_GROUP", StringType(), True),
    StructField("DAYS_UNTIL_EXP", IntegerType(), True),
    StructField("DAYS_SINCE_LAST_RNWL", IntegerType(), True),
    StructField("NUM_OF_RNWLS", IntegerType(), True),
    StructField("HAS_QUOTIENT_ID", IntegerType(), True),
    StructField("L_FIFTY-TWOW_FIRST_MOST_SHOPPED_CATEGORY", StringType(), True),
    StructField("L_FIFTY-TWOW_SECOND_MOST_SHOPPED_CATEGORY", StringType(), True),
    StructField("L52W_PREFERRED_CLUB_NBR", IntegerType(), True),
    StructField("L52W_PREFERRED_CLUB_TRIPS", LongType(), True),
    StructField("L52W_PERCENT_TRIPS_PREFERRED_CLUB", DoubleType(), True),
    StructField("DUMMY_MBR", IntegerType(), True),
    StructField("L12W_SPEND_OVER_P12W_SPEND", DoubleType(), True),
    StructField("L26W_SPEND_OVER_P26W_SPEND", DoubleType(), True),
    StructField("L12W_TRIPS_OVER_P12W_TRIPS", DoubleType(), True),
    StructField("L26W_TRIPS_OVER_P26W_TRIPS", DoubleType(), True),
    StructField("STRATEGIC_MBR_HEADROOM", DoubleType(), True),
    StructField("DIST_RANGE", StringType(), True),
    StructField("AGE_RANGE", StringType(), True),
    StructField("IS_STRATEGIC_MBR", IntegerType(), True),
    StructField("PREFERRED_CLUB_HAS_GAS", IntegerType(), True),
    StructField("mbr_sid", LongType(), True),
    StructField("mbr_prmry_sid", LongType(), True),
    StructField("member_age", DecimalType(10, 0), True),
    StructField("household_income", StringType(), True)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_customer_cube_full,
    primary_keys=["MBRSHP_SID", "FISCAL_WEEK_END"],
    schema=schema,
    description="Member DNA - Merge features"
)

In [0]:
schema = StructType([
    StructField("MBRSHP_SID", LongType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True),
    StructField("FW_ATC_CPN_RED", LongType(), True),
    StructField("FW_ATC_CPN_SPEND", DoubleType(), True),
    StructField("L4W_ATC_CPN_RED", LongType(), True),
    StructField("L4W_ATC_CPN_SPEND", DoubleType(), True),
    StructField("L8W_ATC_CPN_RED", LongType(), True),
    StructField("L8W_ATC_CPN_SPEND", DoubleType(), True),
    StructField("L12W_ATC_CPN_RED", LongType(), True),
    StructField("L12W_ATC_CPN_SPEND", DoubleType(), True),
    StructField("L26W_ATC_CPN_RED", LongType(), True),
    StructField("L26W_ATC_CPN_SPEND", DoubleType(), True),
    StructField("L52W_ATC_CPN_RED", LongType(), True),
    StructField("L52W_ATC_CPN_SPEND", DoubleType(), True),
    StructField("FW_CPN_CLP_COUNT", LongType(), True),
    StructField("L4W_CPN_CLP_COUNT", LongType(), True),
    StructField("L8W_CPN_CLP_COUNT", LongType(), True),
    StructField("L12W_CPN_CLP_COUNT", LongType(), True),
    StructField("L26W_CPN_CLP_COUNT", LongType(), True),
    StructField("L52W_CPN_CLP_COUNT", LongType(), True),
    StructField("FW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("FW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("FW_COUPON_SAVINGS", DoubleType(), True),
    StructField("FW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LAST_FISCAL_WEEK_COUPON_REDEEMED", DateType(), True),
    StructField("LAST_COUPON_REDEEMED", DateType(), True),
    StructField("DAYS_SINCE_LAST_COUPON_REDEEMED", IntegerType(), True),
    StructField("LAST_FISCAL_WEEK_EMAIL_OPENED", DateType(), True),
    StructField("LAST_EMAIL_OPENED", DateType(), True),
    StructField("DAYS_SINCE_LAST_EMAIL_OPENED", IntegerType(), True),
    StructField("LAST_FISCAL_WEEK_ATC_CLIPPED", DateType(), True),
    StructField("LAST_ATC_CLIPPED", DateType(), True),
    StructField("DAYS_SINCE_LAST_ATC_CLIPPED", IntegerType(), True),
    StructField("EMAIL_OPEN_RATE", DoubleType(), True),
    StructField("FW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("cpn_channel", StringType(), True),
    StructField("LFOURW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LEIGHTW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LTWELVEW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LTWENTY-SIXW_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LFIFTY-TWOW_SAVINGS_W_CLPLSS", DoubleType(), True)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_cubes_coupon_and_digital_1,
    primary_keys=["MBRSHP_SID", "FISCAL_WEEK_END"],
    schema=schema,
    description="Member DNA - Coupon and Digital 1"
)

In [0]:
schema = StructType([
    StructField("MBRSHP_SID", LongType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True),
    StructField("LFOURW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("LFOURW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("LFOURW_COUPON_SAVINGS", DoubleType(), True),
    StructField("LFOURW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LEIGHTW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("LEIGHTW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("LEIGHTW_COUPON_SAVINGS", DoubleType(), True),
    StructField("LEIGHTW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LTWELVEW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("LTWELVEW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("LTWELVEW_COUPON_SAVINGS", DoubleType(), True),
    StructField("LTWELVEW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LTWENTY-SIXW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("LTWENTY-SIXW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("LTWENTY-SIXW_COUPON_SAVINGS", DoubleType(), True),
    StructField("LTWENTY-SIXW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True),
    StructField("LFIFTY-TWOW_COUPON_REDEMPTIONS", LongType(), True),
    StructField("LFIFTY-TWOW_COUPON_REDEMPTIONS_W_CLPLSS", LongType(), True),
    StructField("LFIFTY-TWOW_COUPON_SAVINGS", DoubleType(), True),
    StructField("LFIFTY-TWOW_COUPON_SAVINGS_W_CLPLSS", DoubleType(), True)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_cubes_coupon_and_digital_2,
    primary_keys=["MBRSHP_SID", "FISCAL_WEEK_END"],
    schema=schema,
    description="Member DNA - Coupon and Digital 2"
)

In [0]:
schema = StructType([
    StructField("MBRSHP_SID", LongType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True),
    StructField("mbr_sid", LongType(), True),
    StructField("mbr_prmry_sid", LongType(), True),
    StructField("member_age", DecimalType(10, 0), True),
    StructField("household_income", StringType(), True)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_cubes_acquisition,
    primary_keys=["MBRSHP_SID", "FISCAL_WEEK_END"],
    schema=schema,
    description="Member DNA - Acquisition features"
)

In [0]:
schema = StructType([
    StructField("MBRSHP_SID", LongType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True),
    StructField("L_FIFTY-TWOW_FIRST_MOST_SHOPPED_CATEGORY", StringType(), True),
    StructField("L_FIFTY-TWOW_SECOND_MOST_SHOPPED_CATEGORY", StringType(), True)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_cubes_most_shopped,
    primary_keys=["MBRSHP_SID", "FISCAL_WEEK_END"],
    schema=schema,
    description="Member DNA - Most shopped categories"
)

In [0]:
schema = StructType([
    StructField("AH4_CD", StringType(), True),
    StructField("AH4_DESC", StringType(), True),
    StructField("QTY_MBRS", LongType(), True),
    StructField("TRIPS", LongType(), True),
    StructField("UNITS", LongType(), True),
    StructField("SALES", DoubleType(), True),
    StructField("UNIT_RETAIL_PRICE", DoubleType(), True),
    StructField("UNITS_PER_TRIP", DoubleType(), True),
    StructField("UNITS_PER_MBR", DoubleType(), True),
    StructField("TRIPS_PER_MBR", DoubleType(), True),
    StructField("PENETRATION_RATE", DoubleType(), True),
    StructField("QTY_MBRS_1", LongType(), True),
    StructField("QTY_MBRS_2", LongType(), True),
    StructField("QTY_MBRS_3", LongType(), True),
    StructField("QTY_MBRS_4", LongType(), True),
    StructField("QTY_MBRS_5", LongType(), True),
    StructField("QTY_MBRS_6", LongType(), True),
    StructField("QTY_MBRS_7", LongType(), True),
    StructField("QTY_MBRS_8", LongType(), True),
    StructField("QTY_MBRS_9", LongType(), True),
    StructField("QTY_MBRS_10", LongType(), True),
    StructField("QTY_MBRS_11", LongType(), True),
    StructField("QTY_MBRS_12", LongType(), True),
    StructField("MONTH_1_SEASONALITY", DoubleType(), True),
    StructField("MONTH_2_SEASONALITY", DoubleType(), True),
    StructField("MONTH_3_SEASONALITY", DoubleType(), True),
    StructField("MONTH_4_SEASONALITY", DoubleType(), True),
    StructField("MONTH_5_SEASONALITY", DoubleType(), True),
    StructField("MONTH_6_SEASONALITY", DoubleType(), True),
    StructField("MONTH_7_SEASONALITY", DoubleType(), True),
    StructField("MONTH_8_SEASONALITY", DoubleType(), True),
    StructField("MONTH_9_SEASONALITY", DoubleType(), True),
    StructField("MONTH_10_SEASONALITY", DoubleType(), True),
    StructField("MONTH_11_SEASONALITY", DoubleType(), True),
    StructField("MONTH_12_SEASONALITY", DoubleType(), True),
    StructField("PURCHASE_CYCLE_DAYS", DoubleType(), True),
    StructField("PURCHASE_CYCLE", DoubleType(), True),
    StructField("SCALED_MONTH_1_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_2_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_3_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_4_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_5_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_6_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_7_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_8_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_9_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_10_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_11_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_12_SEASONALITY", DoubleType(), True),
    StructField("SCALED_PURCHASE_CYCLE", DoubleType(), True),
    StructField("INCLUDE_OR_EXCLUDE", StringType(), True),
    StructField("EXCLUSION_TYPE", StringType(), True),
    StructField("EXCLUSION_SUBTYPE", StringType(), True),
    StructField("SEASON_MONTH_1", LongType(), True),
    StructField("SEASON_MONTH_2", LongType(), True),
    StructField("SEASON_MONTH_3", LongType(), True),
    StructField("SEASON_MONTH_4", LongType(), True),
    StructField("SEASON_MONTH_5", LongType(), True),
    StructField("SEASON_MONTH_6", LongType(), True),
    StructField("SEASON_MONTH_7", LongType(), True),
    StructField("SEASON_MONTH_8", LongType(), True),
    StructField("SEASON_MONTH_9", LongType(), True),
    StructField("SEASON_MONTH_10", LongType(), True),
    StructField("SEASON_MONTH_11", LongType(), True),
    StructField("SEASON_MONTH_12", LongType(), True),
    StructField("PCT_GROSS_MARGIN", IntegerType(), False),
    StructField("GROSS_MARGIN", IntegerType(), False)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_ah4_cd_category_dna_full,
    primary_keys=["AH4_CD"],
    schema=schema,
    description="Category DNA - AH4 category features"
)

if not spark.catalog.tableExists(ah4_cd_category_dna_archive):    
    spark.sql(f"CREATE TABLE IF NOT EXISTS {ah4_cd_category_dna_archive} LIKE {fs_ah4_cd_category_dna_full}")
    spark.sql(f"ALTER TABLE {ah4_cd_category_dna_archive} ADD COLUMNS (START_DATE DATE, END_DATE DATE, RUN_DATE TIMESTAMP)")

In [0]:
schema = StructType([
    StructField("AH5_CD", StringType(), True),
    StructField("AH5_DESC", StringType(), True),
    StructField("QTY_MBRS", LongType(), True),
    StructField("TRIPS", LongType(), True),
    StructField("UNITS", LongType(), True),
    StructField("SALES", DoubleType(), True),
    StructField("UNIT_RETAIL_PRICE", DoubleType(), True),
    StructField("UNITS_PER_TRIP", DoubleType(), True),
    StructField("UNITS_PER_MBR", DoubleType(), True),
    StructField("TRIPS_PER_MBR", DoubleType(), True),
    StructField("PENETRATION_RATE", DoubleType(), True),
    StructField("QTY_MBRS_1", LongType(), True),
    StructField("QTY_MBRS_2", LongType(), True),
    StructField("QTY_MBRS_3", LongType(), True),
    StructField("QTY_MBRS_4", LongType(), True),
    StructField("QTY_MBRS_5", LongType(), True),
    StructField("QTY_MBRS_6", LongType(), True),
    StructField("QTY_MBRS_7", LongType(), True),
    StructField("QTY_MBRS_8", LongType(), True),
    StructField("QTY_MBRS_9", LongType(), True),
    StructField("QTY_MBRS_10", LongType(), True),
    StructField("QTY_MBRS_11", LongType(), True),
    StructField("QTY_MBRS_12", LongType(), True),
    StructField("MONTH_1_SEASONALITY", DoubleType(), True),
    StructField("MONTH_2_SEASONALITY", DoubleType(), True),
    StructField("MONTH_3_SEASONALITY", DoubleType(), True),
    StructField("MONTH_4_SEASONALITY", DoubleType(), True),
    StructField("MONTH_5_SEASONALITY", DoubleType(), True),
    StructField("MONTH_6_SEASONALITY", DoubleType(), True),
    StructField("MONTH_7_SEASONALITY", DoubleType(), True),
    StructField("MONTH_8_SEASONALITY", DoubleType(), True),
    StructField("MONTH_9_SEASONALITY", DoubleType(), True),
    StructField("MONTH_10_SEASONALITY", DoubleType(), True),
    StructField("MONTH_11_SEASONALITY", DoubleType(), True),
    StructField("MONTH_12_SEASONALITY", DoubleType(), True),
    StructField("PURCHASE_CYCLE_DAYS", DoubleType(), True),
    StructField("PURCHASE_CYCLE", DoubleType(), True),
    StructField("SCALED_MONTH_1_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_2_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_3_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_4_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_5_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_6_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_7_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_8_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_9_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_10_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_11_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_12_SEASONALITY", DoubleType(), True),
    StructField("SCALED_PURCHASE_CYCLE", DoubleType(), True),
    StructField("INCLUDE_OR_EXCLUDE", StringType(), True),
    StructField("EXCLUSION_TYPE", StringType(), True),
    StructField("EXCLUSION_SUBTYPE", StringType(), True),
    StructField("SEASON_MONTH_1", LongType(), True),
    StructField("SEASON_MONTH_2", LongType(), True),
    StructField("SEASON_MONTH_3", LongType(), True),
    StructField("SEASON_MONTH_4", LongType(), True),
    StructField("SEASON_MONTH_5", LongType(), True),
    StructField("SEASON_MONTH_6", LongType(), True),
    StructField("SEASON_MONTH_7", LongType(), True),
    StructField("SEASON_MONTH_8", LongType(), True),
    StructField("SEASON_MONTH_9", LongType(), True),
    StructField("SEASON_MONTH_10", LongType(), True),
    StructField("SEASON_MONTH_11", LongType(), True),
    StructField("SEASON_MONTH_12", LongType(), True),
    StructField("PCT_GROSS_MARGIN", IntegerType(), False),
    StructField("GROSS_MARGIN", IntegerType(), False)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_ah5_cd_category_dna_full,
    primary_keys=["AH5_CD"],
    schema=schema,
    description="Category DNA - AH5 category features"
)

if not spark.catalog.tableExists(ah5_cd_category_dna_archive):    
    spark.sql(f"CREATE TABLE IF NOT EXISTS {ah5_cd_category_dna_archive} LIKE {fs_ah5_cd_category_dna_full}")
    spark.sql(f"ALTER TABLE {ah5_cd_category_dna_archive} ADD COLUMNS (START_DATE DATE, END_DATE DATE, RUN_DATE TIMESTAMP)")

In [0]:
schema = StructType([
    StructField("ARTICLE_NBR", StringType(), True),
    StructField("ARTICLE_DESC", StringType(), True),
    StructField("QTY_MBRS", LongType(), True),
    StructField("TRIPS", LongType(), True),
    StructField("UNITS", LongType(), True),
    StructField("SALES", DoubleType(), True),
    StructField("UNIT_RETAIL_PRICE", DoubleType(), True),
    StructField("UNITS_PER_TRIP", DoubleType(), True),
    StructField("UNITS_PER_MBR", DoubleType(), True),
    StructField("TRIPS_PER_MBR", DoubleType(), True),
    StructField("PENETRATION_RATE", DoubleType(), True),
    StructField("QTY_MBRS_1", LongType(), True),
    StructField("QTY_MBRS_2", LongType(), True),
    StructField("QTY_MBRS_3", LongType(), True),
    StructField("QTY_MBRS_4", LongType(), True),
    StructField("QTY_MBRS_5", LongType(), True),
    StructField("QTY_MBRS_6", LongType(), True),
    StructField("QTY_MBRS_7", LongType(), True),
    StructField("QTY_MBRS_8", LongType(), True),
    StructField("QTY_MBRS_9", LongType(), True),
    StructField("QTY_MBRS_10", LongType(), True),
    StructField("QTY_MBRS_11", LongType(), True),
    StructField("QTY_MBRS_12", LongType(), True),
    StructField("MONTH_1_SEASONALITY", DoubleType(), True),
    StructField("MONTH_2_SEASONALITY", DoubleType(), True),
    StructField("MONTH_3_SEASONALITY", DoubleType(), True),
    StructField("MONTH_4_SEASONALITY", DoubleType(), True),
    StructField("MONTH_5_SEASONALITY", DoubleType(), True),
    StructField("MONTH_6_SEASONALITY", DoubleType(), True),
    StructField("MONTH_7_SEASONALITY", DoubleType(), True),
    StructField("MONTH_8_SEASONALITY", DoubleType(), True),
    StructField("MONTH_9_SEASONALITY", DoubleType(), True),
    StructField("MONTH_10_SEASONALITY", DoubleType(), True),
    StructField("MONTH_11_SEASONALITY", DoubleType(), True),
    StructField("MONTH_12_SEASONALITY", DoubleType(), True),
    StructField("PURCHASE_CYCLE_DAYS", DoubleType(), True),
    StructField("PURCHASE_CYCLE", DoubleType(), True),
    StructField("AH4_CD", StringType(), True),
    StructField("AH4_DESC", StringType(), True),
    StructField("AH5_CD", StringType(), True),
    StructField("AH5_DESC", StringType(), True),
    StructField("SCALED_MONTH_1_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_2_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_3_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_4_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_5_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_6_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_7_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_8_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_9_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_10_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_11_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_12_SEASONALITY", DoubleType(), True),
    StructField("SCALED_PURCHASE_CYCLE", DoubleType(), True),
    StructField("INCLUDE_OR_EXCLUDE", StringType(), True),
    StructField("EXCLUSION_TYPE", StringType(), True),
    StructField("EXCLUSION_SUBTYPE", StringType(), True),
    StructField("SEASON_MONTH_1", LongType(), True),
    StructField("SEASON_MONTH_2", LongType(), True),
    StructField("SEASON_MONTH_3", LongType(), True),
    StructField("SEASON_MONTH_4", LongType(), True),
    StructField("SEASON_MONTH_5", LongType(), True),
    StructField("SEASON_MONTH_6", LongType(), True),
    StructField("SEASON_MONTH_7", LongType(), True),
    StructField("SEASON_MONTH_8", LongType(), True),
    StructField("SEASON_MONTH_9", LongType(), True),
    StructField("SEASON_MONTH_10", LongType(), True),
    StructField("SEASON_MONTH_11", LongType(), True),
    StructField("SEASON_MONTH_12", LongType(), True),
    StructField("PCT_GROSS_MARGIN", IntegerType(), False),
    StructField("GROSS_MARGIN", IntegerType(), False),
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_article_nbr_category_dna_full,
    primary_keys=["ARTICLE_NBR"],
    schema=schema,
    description="Category DNA - Article features"
)

if not spark.catalog.tableExists(article_nbr_category_dna_archive):    
    spark.sql(f"CREATE TABLE IF NOT EXISTS {article_nbr_category_dna_archive} LIKE {fs_article_nbr_category_dna_full}")
    spark.sql(f"ALTER TABLE {article_nbr_category_dna_archive} ADD COLUMNS (START_DATE DATE, END_DATE DATE, RUN_DATE TIMESTAMP)")

In [0]:
schema = StructType([
    StructField("BRAND_CD", IntegerType(), True),
    StructField("BRAND_DESC", StringType(), True),
    StructField("QTY_MBRS", LongType(), True),
    StructField("TRIPS", LongType(), True),
    StructField("UNITS", LongType(), True),
    StructField("SALES", DoubleType(), True),
    StructField("UNIT_RETAIL_PRICE", DoubleType(), True),
    StructField("UNITS_PER_TRIP", DoubleType(), True),
    StructField("UNITS_PER_MBR", DoubleType(), True),
    StructField("TRIPS_PER_MBR", DoubleType(), True),
    StructField("PENETRATION_RATE", DoubleType(), True),
    StructField("QTY_MBRS_1", LongType(), True),
    StructField("QTY_MBRS_2", LongType(), True),
    StructField("QTY_MBRS_3", LongType(), True),
    StructField("QTY_MBRS_4", LongType(), True),
    StructField("QTY_MBRS_5", LongType(), True),
    StructField("QTY_MBRS_6", LongType(), True),
    StructField("QTY_MBRS_7", LongType(), True),
    StructField("QTY_MBRS_8", LongType(), True),
    StructField("QTY_MBRS_9", LongType(), True),
    StructField("QTY_MBRS_10", LongType(), True),
    StructField("QTY_MBRS_11", LongType(), True),
    StructField("QTY_MBRS_12", LongType(), True),
    StructField("MONTH_1_SEASONALITY", DoubleType(), True),
    StructField("MONTH_2_SEASONALITY", DoubleType(), True),
    StructField("MONTH_3_SEASONALITY", DoubleType(), True),
    StructField("MONTH_4_SEASONALITY", DoubleType(), True),
    StructField("MONTH_5_SEASONALITY", DoubleType(), True),
    StructField("MONTH_6_SEASONALITY", DoubleType(), True),
    StructField("MONTH_7_SEASONALITY", DoubleType(), True),
    StructField("MONTH_8_SEASONALITY", DoubleType(), True),
    StructField("MONTH_9_SEASONALITY", DoubleType(), True),
    StructField("MONTH_10_SEASONALITY", DoubleType(), True),
    StructField("MONTH_11_SEASONALITY", DoubleType(), True),
    StructField("MONTH_12_SEASONALITY", DoubleType(), True),
    StructField("PURCHASE_CYCLE_DAYS", DoubleType(), True),
    StructField("PURCHASE_CYCLE", DoubleType(), True),
    StructField("SCALED_MONTH_1_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_2_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_3_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_4_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_5_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_6_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_7_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_8_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_9_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_10_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_11_SEASONALITY", DoubleType(), True),
    StructField("SCALED_MONTH_12_SEASONALITY", DoubleType(), True),
    StructField("SCALED_PURCHASE_CYCLE", DoubleType(), True),
    StructField("INCLUDE_OR_EXCLUDE", StringType(), True),
    StructField("EXCLUSION_TYPE", StringType(), True),
    StructField("EXCLUSION_SUBTYPE", StringType(), True),
    StructField("SEASON_MONTH_1", LongType(), True),
    StructField("SEASON_MONTH_2", LongType(), True),
    StructField("SEASON_MONTH_3", LongType(), True),
    StructField("SEASON_MONTH_4", LongType(), True),
    StructField("SEASON_MONTH_5", LongType(), True),
    StructField("SEASON_MONTH_6", LongType(), True),
    StructField("SEASON_MONTH_7", LongType(), True),
    StructField("SEASON_MONTH_8", LongType(), True),
    StructField("SEASON_MONTH_9", LongType(), True),
    StructField("SEASON_MONTH_10", LongType(), True),
    StructField("SEASON_MONTH_11", LongType(), True),
    StructField("SEASON_MONTH_12", LongType(), True),
    StructField("PCT_GROSS_MARGIN", IntegerType(), False),
    StructField("GROSS_MARGIN", IntegerType(), False)
])

fe = FeatureEngineeringClient()

fe.create_table(
    name=fs_brand_cd_category_dna_full,
    primary_keys=["BRAND_CD"],
    schema=schema,
    description="Category DNA - Brand features"
)

if not spark.catalog.tableExists(brand_cd_category_dna_archive):    
    spark.sql(f"CREATE TABLE IF NOT EXISTS {brand_cd_category_dna_archive} LIKE {fs_brand_cd_category_dna_full}")
    spark.sql(f"ALTER TABLE {brand_cd_category_dna_archive} ADD COLUMNS (START_DATE DATE, END_DATE DATE, RUN_DATE TIMESTAMP)")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {dna_brand_exclusions} (
    CATEGORY_TYPE STRING,
    CATEGORY_CD LONG,
    CATEGORY_DESCRIPTION STRING,
    INCLUDE_OR_EXCLUDE STRING,
    EXCLUSION_TYPE STRING,
    EXCLUSION_SUBTYPE STRING,
    SEASON_MONTH_1 LONG,
    SEASON_MONTH_2 LONG,
    SEASON_MONTH_3 LONG,
    SEASON_MONTH_4 LONG,
    SEASON_MONTH_5 LONG,
    SEASON_MONTH_6 LONG,
    SEASON_MONTH_7 LONG,
    SEASON_MONTH_8 LONG,
    SEASON_MONTH_9 LONG,
    SEASON_MONTH_10 LONG,
    SEASON_MONTH_11 LONG,
    SEASON_MONTH_12 LONG
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {dna_unmatched_brands} (
    BRAND STRING,
    TOP_CAT STRING,
    N_ART LONG,
    N_AH5 LONG,
    MIXED_EXCL INTEGER,
    FY18_SALES DOUBLE,
    TOP_CAT_INCL STRING,
    TOP_CAT_ART_CT LONG,
    PCT_TOP_COVG DOUBLE,
    AH5_DESC STRING,
    EXCLUSION_TYPE STRING,
    EXCLUSION_SUBTYPE STRING,
    SEASON_MONTH_1 LONG,
    SEASON_MONTH_2 LONG,
    SEASON_MONTH_3 LONG,
    SEASON_MONTH_4 LONG,
    SEASON_MONTH_5 LONG,
    SEASON_MONTH_6 LONG,
    SEASON_MONTH_7 LONG,
    SEASON_MONTH_8 LONG,
    SEASON_MONTH_9 LONG,
    SEASON_MONTH_10 LONG,
    SEASON_MONTH_11 LONG,
    SEASON_MONTH_12 LONG,
    INCL_COVG DOUBLE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

### Model

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {model_trip_spend_etl_intermediate} (
    MBRSHP_SID BIGINT NOT NULL,
    FISCAL_WEEK_END DATE NOT NULL,
    FISCAL_WEEK_START DATE,
    FISCAL_L4W_END DATE,
    FISCAL_L8W_END DATE,
    FISCAL_L12W_END DATE,
    FISCAL_L26W_END DATE,
    FISCAL_L52W_END DATE,
    L52W_G4W_STDEV_TRIPS DOUBLE NOT NULL,
    L52W_G4W_STDEV_SPEND DOUBLE NOT NULL,
    WEEK_TRIPS BIGINT NOT NULL,
    LAST_FOUR_WEEK_TRIPS BIGINT NOT NULL,
    LAST_EIGHT_WEEK_TRIPS BIGINT NOT NULL,
    LAST_TWELVE_WEEK_TRIPS BIGINT NOT NULL,
    `LAST_TWENTY-SIX_WEEK_TRIPS` BIGINT NOT NULL,
    `LAST_FIFTY-TWO_WEEK_TRIPS` BIGINT NOT NULL,
    WEEK_SPEND DOUBLE NOT NULL,
    FW_SPEND_IN_STORE DOUBLE NOT NULL,
    WEEK_UNITS BIGINT NOT NULL,
    WEEK_UNITS_OVER_FIFTY BIGINT NOT NULL,
    FW_GAS_TRIPS BIGINT NOT NULL,
    FW_GAS_DISTINCT_DAYS BIGINT NOT NULL,
    FW_GAS_SPEND DOUBLE NOT NULL,
    FW_GAS_AND_STORE_DISTINCT_DAYS BIGINT NOT NULL,
    FW_ECOMMERCE_SPEND DOUBLE NOT NULL,
    FW_ECOMMERCE_TRIPS BIGINT NOT NULL,
    WEEK_DISTINCT_DAYS BIGINT NOT NULL,
    LAST_FOUR_WEEK_DISTINCT_DAYS BIGINT NOT NULL,
    LAST_EIGHT_WEEK_DISTINCT_DAYS BIGINT NOT NULL,
    LAST_TWELVE_WEEK_DISTINCT_DAYS BIGINT NOT NULL,
    `LAST_TWENTY-SIX_WEEK_DISTINCT_DAYS` BIGINT NOT NULL,
    `LAST_FIFTY-TWO_WEEK_DISTINCT_DAYS` BIGINT NOT NULL,
    WEEK_TRANSACTIONS BIGINT NOT NULL,
    LAST_FOUR_WEEK_SPEND DOUBLE NOT NULL,
    LAST_EIGHT_WEEK_SPEND DOUBLE NOT NULL,
    LAST_TWELVE_WEEK_SPEND DOUBLE NOT NULL,
    `LAST_TWENTY-SIX_WEEK_SPEND` DOUBLE NOT NULL,
    `LAST_FIFTY-TWO_WEEK_SPEND` DOUBLE NOT NULL,
    LFOURW_SPEND_IN_STORE DOUBLE NOT NULL,
    LEIGHTW_SPEND_IN_STORE DOUBLE NOT NULL,
    LTWELVEW_SPEND_IN_STORE DOUBLE NOT NULL,
    `LTWENTY-SIXW_SPEND_IN_STORE` DOUBLE NOT NULL,
    `LFIFTY-TWOW_SPEND_IN_STORE` DOUBLE NOT NULL,
    LAST_FOUR_WEEK_UNITS BIGINT NOT NULL,
    LAST_EIGHT_WEEK_UNITS BIGINT NOT NULL,
    LAST_TWELVE_WEEK_UNITS BIGINT NOT NULL,
    `LAST_TWENTY-SIX_WEEK_UNITS` BIGINT NOT NULL,
    `LAST_FIFTY-TWO_WEEK_UNITS` BIGINT NOT NULL,
    LAST_FOUR_WEEK_UNITS_OVER_FIFTY BIGINT NOT NULL,
    LAST_EIGHT_WEEK_UNITS_OVER_FIFTY BIGINT NOT NULL,
    LAST_TWELVE_WEEK_UNITS_OVER_FIFTY BIGINT NOT NULL,
    `LAST_TWENTY-SIX_WEEK_UNITS_OVER_FIFTY` BIGINT NOT NULL,
    `LAST_FIFTY-TWO_WEEK_UNITS_OVER_FIFTY` BIGINT NOT NULL,
    L_FOURW_GAS_TRIPS BIGINT NOT NULL,
    L_EIGHTW_GAS_TRIPS BIGINT NOT NULL,
    L_TWELVEW_GAS_TRIPS BIGINT NOT NULL,
    `L_TWENTY-SIXW_GAS_TRIPS` BIGINT NOT NULL,
    `L_FIFTY-TWOW_GAS_TRIPS` BIGINT NOT NULL,
    L_FOURW_GAS_DISTINCT_DAYS BIGINT NOT NULL,
    L_EIGHTW_GAS_DISTINCT_DAYS BIGINT NOT NULL,
    L_TWELVEW_GAS_DISTINCT_DAYS BIGINT NOT NULL,
    `L_TWENTY-SIXW_GAS_DISTINCT_DAYS` BIGINT NOT NULL,
    `L_FIFTY-TWOW_GAS_DISTINCT_DAYS` BIGINT NOT NULL,
    L_FOURW_GAS_SPEND DOUBLE NOT NULL,
    L_EIGHTW_GAS_SPEND DOUBLE NOT NULL,
    L_TWELVEW_GAS_SPEND DOUBLE NOT NULL,
    `L_TWENTY-SIXW_GAS_SPEND` DOUBLE NOT NULL,
    `L_FIFTY-TWOW_GAS_SPEND` DOUBLE NOT NULL,
    L_FOURW_GAS_AND_STORE_DISTINCT_DAYS BIGINT NOT NULL,
    L_EIGHTW_GAS_AND_STORE_DISTINCT_DAYS BIGINT NOT NULL,
    L_TWELVEW_GAS_AND_STORE_DISTINCT_DAYS BIGINT NOT NULL,
    `L_TWENTY-SIXW_GAS_AND_STORE_DISTINCT_DAYS` BIGINT NOT NULL,
    `L_FIFTY-TWOW_GAS_AND_STORE_DISTINCT_DAYS` BIGINT NOT NULL,
    L_FOURW_ECOMMERCE_SPEND DOUBLE NOT NULL,
    L_EIGHTW_ECOMMERCE_SPEND DOUBLE NOT NULL,
    L_TWELVEW_ECOMMERCE_SPEND DOUBLE NOT NULL,
    `L_TWENTY-SIXW_ECOMMERCE_SPEND` DOUBLE NOT NULL,
    `L_FIFTY-TWOW_ECOMMERCE_SPEND` DOUBLE NOT NULL,
    L_FOURW_ECOMMERCE_TRIPS BIGINT NOT NULL,
    L_EIGHTW_ECOMMERCE_TRIPS BIGINT NOT NULL,
    L_TWELVEW_ECOMMERCE_TRIPS BIGINT NOT NULL,
    `L_TWENTY-SIXW_ECOMMERCE_TRIPS` BIGINT NOT NULL,
    `L_FIFTY-TWOW_ECOMMERCE_TRIPS` BIGINT NOT NULL,
    LAST_FOUR_WEEK_TRANSACTIONS BIGINT NOT NULL,
    LAST_EIGHT_WEEK_TRANSACTIONS BIGINT NOT NULL,
    LAST_TWELVE_WEEK_TRANSACTIONS BIGINT NOT NULL,
    `LAST_TWENTY-SIX_WEEK_TRANSACTIONS` BIGINT NOT NULL,
    `LAST_FIFTY-TWO_WEEK_TRANSACTIONS` BIGINT NOT NULL,
    L52W_CREDIT DECIMAL(38,2) NOT NULL,
    L52W_DEBIT DECIMAL(38,2) NOT NULL,
    L52W_CASH DECIMAL(38,2) NOT NULL,
    L52W_COUPON DECIMAL(38,2) NOT NULL,
    L52W_EBT DECIMAL(38,2) NOT NULL,
    L52W_BJS DECIMAL(38,2) NOT NULL,
    FW_HAS_BOUGHT_MEN INT NOT NULL,
    L52W_HAS_BOUGHT_MEN INT NOT NULL,
    FW_HAS_BOUGHT_WOMEN INT NOT NULL,
    L52W_HAS_BOUGHT_WOMEN INT NOT NULL,
    FW_HAS_BOUGHT_PET INT NOT NULL,
    L52W_HAS_BOUGHT_PET INT NOT NULL,
    FW_HAS_BOUGHT_CHILDREN INT NOT NULL,
    L52W_HAS_BOUGHT_CHILDREN INT NOT NULL,
    FW_HAS_BOUGHT_BABY INT NOT NULL,
    L52W_HAS_BOUGHT_BABY INT NOT NULL,
    L52W_MEDIAN_BASKETSIZE FLOAT NOT NULL,
    L52W_DISTINCT_CATEGORIES BIGINT NOT NULL,
    LAST_FISCAL_WEEK_TRIP DATE,
    LAST_TRIP DATE,
    DAYS_SINCE_LAST_TRIP INT NOT NULL,
    L52W_MAX_INTERVAL INT NOT NULL,
    L52W_MIN_INTERVAL INT NOT NULL,
    FW_ATC_CPN_RED BIGINT NOT NULL,
    FW_ATC_CPN_SPEND DOUBLE NOT NULL,
    L4W_ATC_CPN_RED BIGINT NOT NULL,
    L4W_ATC_CPN_SPEND DOUBLE NOT NULL,
    L8W_ATC_CPN_RED BIGINT NOT NULL,
    L8W_ATC_CPN_SPEND DOUBLE NOT NULL,
    L12W_ATC_CPN_RED BIGINT NOT NULL,
    L12W_ATC_CPN_SPEND DOUBLE NOT NULL,
    L26W_ATC_CPN_RED BIGINT NOT NULL,
    L26W_ATC_CPN_SPEND DOUBLE NOT NULL,
    L52W_ATC_CPN_RED BIGINT NOT NULL,
    L52W_ATC_CPN_SPEND DOUBLE NOT NULL,
    FW_CPN_CLP_COUNT BIGINT NOT NULL,
    L4W_CPN_CLP_COUNT BIGINT NOT NULL,
    L8W_CPN_CLP_COUNT BIGINT NOT NULL,
    L12W_CPN_CLP_COUNT BIGINT NOT NULL,
    L26W_CPN_CLP_COUNT BIGINT NOT NULL,
    L52W_CPN_CLP_COUNT BIGINT NOT NULL,
    FW_COUPON_REDEMPTIONS BIGINT NOT NULL,
    FW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT NOT NULL,
    FW_COUPON_SAVINGS DOUBLE NOT NULL,
    FW_COUPON_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    LAST_FISCAL_WEEK_COUPON_REDEEMED DATE,
    LAST_COUPON_REDEEMED DATE,
    DAYS_SINCE_LAST_COUPON_REDEEMED INT NOT NULL,
    LAST_FISCAL_WEEK_EMAIL_OPENED DATE,
    LAST_EMAIL_OPENED DATE,
    DAYS_SINCE_LAST_EMAIL_OPENED INT NOT NULL,
    LAST_FISCAL_WEEK_ATC_CLIPPED DATE,
    LAST_ATC_CLIPPED DATE,
    DAYS_SINCE_LAST_ATC_CLIPPED INT NOT NULL,
    EMAIL_OPEN_RATE DOUBLE NOT NULL,
    FW_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    cpn_channel STRING,
    LFOURW_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    LEIGHTW_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    LTWELVEW_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    `LTWENTY-SIXW_SAVINGS_W_CLPLSS` DOUBLE NOT NULL,
    `LFIFTY-TWOW_SAVINGS_W_CLPLSS` DOUBLE NOT NULL,
    LFOURW_COUPON_REDEMPTIONS BIGINT NOT NULL,
    LFOURW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT NOT NULL,
    LFOURW_COUPON_SAVINGS DOUBLE NOT NULL,
    LFOURW_COUPON_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    LEIGHTW_COUPON_REDEMPTIONS BIGINT NOT NULL,
    LEIGHTW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT NOT NULL,
    LEIGHTW_COUPON_SAVINGS DOUBLE NOT NULL,
    LEIGHTW_COUPON_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    LTWELVEW_COUPON_REDEMPTIONS BIGINT NOT NULL,
    LTWELVEW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT NOT NULL,
    LTWELVEW_COUPON_SAVINGS DOUBLE NOT NULL,
    LTWELVEW_COUPON_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    `LTWENTY-SIXW_COUPON_REDEMPTIONS` BIGINT NOT NULL,
    `LTWENTY-SIXW_COUPON_REDEMPTIONS_W_CLPLSS` BIGINT NOT NULL,
    `LTWENTY-SIXW_COUPON_SAVINGS` DOUBLE NOT NULL,
    `LTWENTY-SIXW_COUPON_SAVINGS_W_CLPLSS` DOUBLE NOT NULL,
    `LFIFTY-TWOW_COUPON_REDEMPTIONS` BIGINT NOT NULL,
    `LFIFTY-TWOW_COUPON_REDEMPTIONS_W_CLPLSS` BIGINT NOT NULL,
    `LFIFTY-TWOW_COUPON_SAVINGS` DOUBLE NOT NULL,
    `LFIFTY-TWOW_COUPON_SAVINGS_W_CLPLSS` DOUBLE NOT NULL,
    LATEST_MBRSHP_NBR STRING,
    ZIP STRING,
    EFF_DT DATE,
    LATEST_MBRSHP_TYPE_ID STRING,
    LATEST_MBRSHP_FEE_INC DECIMAL(5,2),
    LATEST_MBRSHP_SUB_TYPE STRING,
    LATEST_MBRSHP_ENR_DT DATE,
    LATEST_MBRSHP_EXP_DT DATE,
    LATEST_MBRSHP_RNWL_DT DATE,
    LATEST_RWDS_MBR_IND STRING,
    LATEST_MKT_CD STRING,
    LATEST_HOME_ZIP_CD STRING,
    LATEST_SIC_CD INT NOT NULL,
    LATEST_CLUB_OF_FREQUENCY INT NOT NULL,
    LATEST_PRI_SUPP_FHH_IND INT NOT NULL,
    LATEST_GRP_AFFIL_ID STRING,
    LATEST_ER_SIGNUP_DT DATE,
    LATEST_AUTO_RNWL_IND STRING,
    LATEST_TM_MBR_IND INT NOT NULL,
    LATEST_TRIAL_MBR_IND INT NOT NULL,
    LATEST_MFI_TIER INT NOT NULL,
    EXP_DT DATE,
    MBRSHP_STAT_CD STRING,
    MBRSHP_EXP_DT DATE,
    MBRSHP_RNWL_DT DATE,
    MBRSHP_FEE_INC DOUBLE NOT NULL,
    RWDS_MBR_IND STRING,
    RWDS_MBR_ENR_DT DATE,
    CLUB_OF_FREQUENCY INT NOT NULL,
    TEAM_MBR_IND STRING,
    FIRST_MBRSHP_FEE_INC DOUBLE NOT NULL,
    ZIP_DISTANCE DOUBLE NOT NULL,
    BJS_DRIVING_DISTANCE DOUBLE NOT NULL,
    BJS_DISTANCE DOUBLE NOT NULL,
    BJS_DRIVE_TIME DOUBLE NOT NULL,
    WALMART_DRIVE_TIME DOUBLE NOT NULL,
    WALMART_DRIVING_DISTANCE DOUBLE NOT NULL,
    WALMART_DISTANCE DOUBLE NOT NULL,
    COSTCO_DRIVE_TIME DOUBLE NOT NULL,
    COSTCO_DRIVING_DISTANCE DOUBLE NOT NULL,
    COSTCO_DISTANCE DOUBLE NOT NULL,
    SAMS_DRIVE_TIME DOUBLE NOT NULL,
    SAMS_DRIVING_DISTANCE DOUBLE NOT NULL,
    SAMS_DISTANCE DOUBLE NOT NULL,
    TENURE INT NOT NULL,
    TENURE_GROUP STRING,
    DAYS_UNTIL_EXP INT NOT NULL,
    DAYS_SINCE_LAST_RNWL INT NOT NULL,
    NUM_OF_RNWLS INT NOT NULL,
    HAS_QUOTIENT_ID INT NOT NULL,
    `L_FIFTY-TWOW_FIRST_MOST_SHOPPED_CATEGORY` STRING,
    `L_FIFTY-TWOW_SECOND_MOST_SHOPPED_CATEGORY` STRING,
    L52W_PREFERRED_CLUB_NBR INT NOT NULL,
    L52W_PREFERRED_CLUB_TRIPS BIGINT NOT NULL,
    L52W_PERCENT_TRIPS_PREFERRED_CLUB DOUBLE NOT NULL,
    DUMMY_MBR INT NOT NULL,
    L12W_SPEND_OVER_P12W_SPEND DOUBLE NOT NULL,
    L26W_SPEND_OVER_P26W_SPEND DOUBLE NOT NULL,
    L12W_TRIPS_OVER_P12W_TRIPS DOUBLE NOT NULL,
    L26W_TRIPS_OVER_P26W_TRIPS DOUBLE NOT NULL,
    STRATEGIC_MBR_HEADROOM DOUBLE NOT NULL,
    DIST_RANGE STRING,
    AGE_RANGE STRING,
    IS_STRATEGIC_MBR INT NOT NULL,
    PREFERRED_CLUB_HAS_GAS INT NOT NULL,
    mbr_sid BIGINT NOT NULL,
    mbr_prmry_sid BIGINT NOT NULL,
    member_age DECIMAL(10,0),
    household_income STRING,
    MEMBER_FREQUENCY_GROUP STRING NOT NULL,
    `SPEND_IN_STORE_BY_TRIPS_LAST_TWENTY-SIX_WEEKS` DOUBLE NOT NULL,
    WEEK_OF_YEAR INT NOT NULL,
    MONTH INT NOT NULL,
    will_visit_from_0_2 INT NOT NULL,
    will_visit_from_3_5 INT NOT NULL,
    will_visit_from_5_7 INT NOT NULL,
    spend_from_0_2 DOUBLE NOT NULL,
    spend_from_3_5 DOUBLE NOT NULL,
    spend_from_5_7 DOUBLE NOT NULL
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {model_trip_spend_etl_intermediate_archive} (
    MBRSHP_SID BIGINT NOT NULL,
    FISCAL_WEEK_END DATE NOT NULL,
    FISCAL_WEEK_START DATE,
    FISCAL_L4W_END DATE,
    FISCAL_L8W_END DATE,
    FISCAL_L12W_END DATE,
    FISCAL_L26W_END DATE,
    FISCAL_L52W_END DATE,
    L52W_G4W_STDEV_TRIPS DOUBLE NOT NULL,
    L52W_G4W_STDEV_SPEND DOUBLE NOT NULL,
    WEEK_TRIPS BIGINT NOT NULL,
    LAST_FOUR_WEEK_TRIPS BIGINT NOT NULL,
    LAST_EIGHT_WEEK_TRIPS BIGINT NOT NULL,
    LAST_TWELVE_WEEK_TRIPS BIGINT NOT NULL,
    `LAST_TWENTY-SIX_WEEK_TRIPS` BIGINT NOT NULL,
    `LAST_FIFTY-TWO_WEEK_TRIPS` BIGINT NOT NULL,
    WEEK_SPEND DOUBLE NOT NULL,
    FW_SPEND_IN_STORE DOUBLE NOT NULL,
    WEEK_UNITS BIGINT NOT NULL,
    WEEK_UNITS_OVER_FIFTY BIGINT NOT NULL,
    FW_GAS_TRIPS BIGINT NOT NULL,
    FW_GAS_DISTINCT_DAYS BIGINT NOT NULL,
    FW_GAS_SPEND DOUBLE NOT NULL,
    FW_GAS_AND_STORE_DISTINCT_DAYS BIGINT NOT NULL,
    FW_ECOMMERCE_SPEND DOUBLE NOT NULL,
    FW_ECOMMERCE_TRIPS BIGINT NOT NULL,
    WEEK_DISTINCT_DAYS BIGINT NOT NULL,
    LAST_FOUR_WEEK_DISTINCT_DAYS BIGINT NOT NULL,
    LAST_EIGHT_WEEK_DISTINCT_DAYS BIGINT NOT NULL,
    LAST_TWELVE_WEEK_DISTINCT_DAYS BIGINT NOT NULL,
    `LAST_TWENTY-SIX_WEEK_DISTINCT_DAYS` BIGINT NOT NULL,
    `LAST_FIFTY-TWO_WEEK_DISTINCT_DAYS` BIGINT NOT NULL,
    WEEK_TRANSACTIONS BIGINT NOT NULL,
    LAST_FOUR_WEEK_SPEND DOUBLE NOT NULL,
    LAST_EIGHT_WEEK_SPEND DOUBLE NOT NULL,
    LAST_TWELVE_WEEK_SPEND DOUBLE NOT NULL,
    `LAST_TWENTY-SIX_WEEK_SPEND` DOUBLE NOT NULL,
    `LAST_FIFTY-TWO_WEEK_SPEND` DOUBLE NOT NULL,
    LFOURW_SPEND_IN_STORE DOUBLE NOT NULL,
    LEIGHTW_SPEND_IN_STORE DOUBLE NOT NULL,
    LTWELVEW_SPEND_IN_STORE DOUBLE NOT NULL,
    `LTWENTY-SIXW_SPEND_IN_STORE` DOUBLE NOT NULL,
    `LFIFTY-TWOW_SPEND_IN_STORE` DOUBLE NOT NULL,
    LAST_FOUR_WEEK_UNITS BIGINT NOT NULL,
    LAST_EIGHT_WEEK_UNITS BIGINT NOT NULL,
    LAST_TWELVE_WEEK_UNITS BIGINT NOT NULL,
    `LAST_TWENTY-SIX_WEEK_UNITS` BIGINT NOT NULL,
    `LAST_FIFTY-TWO_WEEK_UNITS` BIGINT NOT NULL,
    LAST_FOUR_WEEK_UNITS_OVER_FIFTY BIGINT NOT NULL,
    LAST_EIGHT_WEEK_UNITS_OVER_FIFTY BIGINT NOT NULL,
    LAST_TWELVE_WEEK_UNITS_OVER_FIFTY BIGINT NOT NULL,
    `LAST_TWENTY-SIX_WEEK_UNITS_OVER_FIFTY` BIGINT NOT NULL,
    `LAST_FIFTY-TWO_WEEK_UNITS_OVER_FIFTY` BIGINT NOT NULL,
    L_FOURW_GAS_TRIPS BIGINT NOT NULL,
    L_EIGHTW_GAS_TRIPS BIGINT NOT NULL,
    L_TWELVEW_GAS_TRIPS BIGINT NOT NULL,
    `L_TWENTY-SIXW_GAS_TRIPS` BIGINT NOT NULL,
    `L_FIFTY-TWOW_GAS_TRIPS` BIGINT NOT NULL,
    L_FOURW_GAS_DISTINCT_DAYS BIGINT NOT NULL,
    L_EIGHTW_GAS_DISTINCT_DAYS BIGINT NOT NULL,
    L_TWELVEW_GAS_DISTINCT_DAYS BIGINT NOT NULL,
    `L_TWENTY-SIXW_GAS_DISTINCT_DAYS` BIGINT NOT NULL,
    `L_FIFTY-TWOW_GAS_DISTINCT_DAYS` BIGINT NOT NULL,
    L_FOURW_GAS_SPEND DOUBLE NOT NULL,
    L_EIGHTW_GAS_SPEND DOUBLE NOT NULL,
    L_TWELVEW_GAS_SPEND DOUBLE NOT NULL,
    `L_TWENTY-SIXW_GAS_SPEND` DOUBLE NOT NULL,
    `L_FIFTY-TWOW_GAS_SPEND` DOUBLE NOT NULL,
    L_FOURW_GAS_AND_STORE_DISTINCT_DAYS BIGINT NOT NULL,
    L_EIGHTW_GAS_AND_STORE_DISTINCT_DAYS BIGINT NOT NULL,
    L_TWELVEW_GAS_AND_STORE_DISTINCT_DAYS BIGINT NOT NULL,
    `L_TWENTY-SIXW_GAS_AND_STORE_DISTINCT_DAYS` BIGINT NOT NULL,
    `L_FIFTY-TWOW_GAS_AND_STORE_DISTINCT_DAYS` BIGINT NOT NULL,
    L_FOURW_ECOMMERCE_SPEND DOUBLE NOT NULL,
    L_EIGHTW_ECOMMERCE_SPEND DOUBLE NOT NULL,
    L_TWELVEW_ECOMMERCE_SPEND DOUBLE NOT NULL,
    `L_TWENTY-SIXW_ECOMMERCE_SPEND` DOUBLE NOT NULL,
    `L_FIFTY-TWOW_ECOMMERCE_SPEND` DOUBLE NOT NULL,
    L_FOURW_ECOMMERCE_TRIPS BIGINT NOT NULL,
    L_EIGHTW_ECOMMERCE_TRIPS BIGINT NOT NULL,
    L_TWELVEW_ECOMMERCE_TRIPS BIGINT NOT NULL,
    `L_TWENTY-SIXW_ECOMMERCE_TRIPS` BIGINT NOT NULL,
    `L_FIFTY-TWOW_ECOMMERCE_TRIPS` BIGINT NOT NULL,
    LAST_FOUR_WEEK_TRANSACTIONS BIGINT NOT NULL,
    LAST_EIGHT_WEEK_TRANSACTIONS BIGINT NOT NULL,
    LAST_TWELVE_WEEK_TRANSACTIONS BIGINT NOT NULL,
    `LAST_TWENTY-SIX_WEEK_TRANSACTIONS` BIGINT NOT NULL,
    `LAST_FIFTY-TWO_WEEK_TRANSACTIONS` BIGINT NOT NULL,
    L52W_CREDIT DECIMAL(38,2) NOT NULL,
    L52W_DEBIT DECIMAL(38,2) NOT NULL,
    L52W_CASH DECIMAL(38,2) NOT NULL,
    L52W_COUPON DECIMAL(38,2) NOT NULL,
    L52W_EBT DECIMAL(38,2) NOT NULL,
    L52W_BJS DECIMAL(38,2) NOT NULL,
    FW_HAS_BOUGHT_MEN INT NOT NULL,
    L52W_HAS_BOUGHT_MEN INT NOT NULL,
    FW_HAS_BOUGHT_WOMEN INT NOT NULL,
    L52W_HAS_BOUGHT_WOMEN INT NOT NULL,
    FW_HAS_BOUGHT_PET INT NOT NULL,
    L52W_HAS_BOUGHT_PET INT NOT NULL,
    FW_HAS_BOUGHT_CHILDREN INT NOT NULL,
    L52W_HAS_BOUGHT_CHILDREN INT NOT NULL,
    FW_HAS_BOUGHT_BABY INT NOT NULL,
    L52W_HAS_BOUGHT_BABY INT NOT NULL,
    L52W_MEDIAN_BASKETSIZE FLOAT NOT NULL,
    L52W_DISTINCT_CATEGORIES BIGINT NOT NULL,
    LAST_FISCAL_WEEK_TRIP DATE,
    LAST_TRIP DATE,
    DAYS_SINCE_LAST_TRIP INT NOT NULL,
    L52W_MAX_INTERVAL INT NOT NULL,
    L52W_MIN_INTERVAL INT NOT NULL,
    FW_ATC_CPN_RED BIGINT NOT NULL,
    FW_ATC_CPN_SPEND DOUBLE NOT NULL,
    L4W_ATC_CPN_RED BIGINT NOT NULL,
    L4W_ATC_CPN_SPEND DOUBLE NOT NULL,
    L8W_ATC_CPN_RED BIGINT NOT NULL,
    L8W_ATC_CPN_SPEND DOUBLE NOT NULL,
    L12W_ATC_CPN_RED BIGINT NOT NULL,
    L12W_ATC_CPN_SPEND DOUBLE NOT NULL,
    L26W_ATC_CPN_RED BIGINT NOT NULL,
    L26W_ATC_CPN_SPEND DOUBLE NOT NULL,
    L52W_ATC_CPN_RED BIGINT NOT NULL,
    L52W_ATC_CPN_SPEND DOUBLE NOT NULL,
    FW_CPN_CLP_COUNT BIGINT NOT NULL,
    L4W_CPN_CLP_COUNT BIGINT NOT NULL,
    L8W_CPN_CLP_COUNT BIGINT NOT NULL,
    L12W_CPN_CLP_COUNT BIGINT NOT NULL,
    L26W_CPN_CLP_COUNT BIGINT NOT NULL,
    L52W_CPN_CLP_COUNT BIGINT NOT NULL,
    FW_COUPON_REDEMPTIONS BIGINT NOT NULL,
    FW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT NOT NULL,
    FW_COUPON_SAVINGS DOUBLE NOT NULL,
    FW_COUPON_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    LAST_FISCAL_WEEK_COUPON_REDEEMED DATE,
    LAST_COUPON_REDEEMED DATE,
    DAYS_SINCE_LAST_COUPON_REDEEMED INT NOT NULL,
    LAST_FISCAL_WEEK_EMAIL_OPENED DATE,
    LAST_EMAIL_OPENED DATE,
    DAYS_SINCE_LAST_EMAIL_OPENED INT NOT NULL,
    LAST_FISCAL_WEEK_ATC_CLIPPED DATE,
    LAST_ATC_CLIPPED DATE,
    DAYS_SINCE_LAST_ATC_CLIPPED INT NOT NULL,
    EMAIL_OPEN_RATE DOUBLE NOT NULL,
    FW_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    cpn_channel STRING,
    LFOURW_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    LEIGHTW_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    LTWELVEW_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    `LTWENTY-SIXW_SAVINGS_W_CLPLSS` DOUBLE NOT NULL,
    `LFIFTY-TWOW_SAVINGS_W_CLPLSS` DOUBLE NOT NULL,
    LFOURW_COUPON_REDEMPTIONS BIGINT NOT NULL,
    LFOURW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT NOT NULL,
    LFOURW_COUPON_SAVINGS DOUBLE NOT NULL,
    LFOURW_COUPON_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    LEIGHTW_COUPON_REDEMPTIONS BIGINT NOT NULL,
    LEIGHTW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT NOT NULL,
    LEIGHTW_COUPON_SAVINGS DOUBLE NOT NULL,
    LEIGHTW_COUPON_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    LTWELVEW_COUPON_REDEMPTIONS BIGINT NOT NULL,
    LTWELVEW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT NOT NULL,
    LTWELVEW_COUPON_SAVINGS DOUBLE NOT NULL,
    LTWELVEW_COUPON_SAVINGS_W_CLPLSS DOUBLE NOT NULL,
    `LTWENTY-SIXW_COUPON_REDEMPTIONS` BIGINT NOT NULL,
    `LTWENTY-SIXW_COUPON_REDEMPTIONS_W_CLPLSS` BIGINT NOT NULL,
    `LTWENTY-SIXW_COUPON_SAVINGS` DOUBLE NOT NULL,
    `LTWENTY-SIXW_COUPON_SAVINGS_W_CLPLSS` DOUBLE NOT NULL,
    `LFIFTY-TWOW_COUPON_REDEMPTIONS` BIGINT NOT NULL,
    `LFIFTY-TWOW_COUPON_REDEMPTIONS_W_CLPLSS` BIGINT NOT NULL,
    `LFIFTY-TWOW_COUPON_SAVINGS` DOUBLE NOT NULL,
    `LFIFTY-TWOW_COUPON_SAVINGS_W_CLPLSS` DOUBLE NOT NULL,
    LATEST_MBRSHP_NBR STRING,
    ZIP STRING,
    EFF_DT DATE,
    LATEST_MBRSHP_TYPE_ID STRING,
    LATEST_MBRSHP_FEE_INC DECIMAL(5,2),
    LATEST_MBRSHP_SUB_TYPE STRING,
    LATEST_MBRSHP_ENR_DT DATE,
    LATEST_MBRSHP_EXP_DT DATE,
    LATEST_MBRSHP_RNWL_DT DATE,
    LATEST_RWDS_MBR_IND STRING,
    LATEST_MKT_CD STRING,
    LATEST_HOME_ZIP_CD STRING,
    LATEST_SIC_CD INT NOT NULL,
    LATEST_CLUB_OF_FREQUENCY INT NOT NULL,
    LATEST_PRI_SUPP_FHH_IND INT NOT NULL,
    LATEST_GRP_AFFIL_ID STRING,
    LATEST_ER_SIGNUP_DT DATE,
    LATEST_AUTO_RNWL_IND STRING,
    LATEST_TM_MBR_IND INT NOT NULL,
    LATEST_TRIAL_MBR_IND INT NOT NULL,
    LATEST_MFI_TIER INT NOT NULL,
    EXP_DT DATE,
    MBRSHP_STAT_CD STRING,
    MBRSHP_EXP_DT DATE,
    MBRSHP_RNWL_DT DATE,
    MBRSHP_FEE_INC DOUBLE NOT NULL,
    RWDS_MBR_IND STRING,
    RWDS_MBR_ENR_DT DATE,
    CLUB_OF_FREQUENCY INT NOT NULL,
    TEAM_MBR_IND STRING,
    FIRST_MBRSHP_FEE_INC DOUBLE NOT NULL,
    ZIP_DISTANCE DOUBLE NOT NULL,
    BJS_DRIVING_DISTANCE DOUBLE NOT NULL,
    BJS_DISTANCE DOUBLE NOT NULL,
    BJS_DRIVE_TIME DOUBLE NOT NULL,
    WALMART_DRIVE_TIME DOUBLE NOT NULL,
    WALMART_DRIVING_DISTANCE DOUBLE NOT NULL,
    WALMART_DISTANCE DOUBLE NOT NULL,
    COSTCO_DRIVE_TIME DOUBLE NOT NULL,
    COSTCO_DRIVING_DISTANCE DOUBLE NOT NULL,
    COSTCO_DISTANCE DOUBLE NOT NULL,
    SAMS_DRIVE_TIME DOUBLE NOT NULL,
    SAMS_DRIVING_DISTANCE DOUBLE NOT NULL,
    SAMS_DISTANCE DOUBLE NOT NULL,
    TENURE INT NOT NULL,
    TENURE_GROUP STRING,
    DAYS_UNTIL_EXP INT NOT NULL,
    DAYS_SINCE_LAST_RNWL INT NOT NULL,
    NUM_OF_RNWLS INT NOT NULL,
    HAS_QUOTIENT_ID INT NOT NULL,
    `L_FIFTY-TWOW_FIRST_MOST_SHOPPED_CATEGORY` STRING,
    `L_FIFTY-TWOW_SECOND_MOST_SHOPPED_CATEGORY` STRING,
    L52W_PREFERRED_CLUB_NBR INT NOT NULL,
    L52W_PREFERRED_CLUB_TRIPS BIGINT NOT NULL,
    L52W_PERCENT_TRIPS_PREFERRED_CLUB DOUBLE NOT NULL,
    DUMMY_MBR INT NOT NULL,
    L12W_SPEND_OVER_P12W_SPEND DOUBLE NOT NULL,
    L26W_SPEND_OVER_P26W_SPEND DOUBLE NOT NULL,
    L12W_TRIPS_OVER_P12W_TRIPS DOUBLE NOT NULL,
    L26W_TRIPS_OVER_P26W_TRIPS DOUBLE NOT NULL,
    STRATEGIC_MBR_HEADROOM DOUBLE NOT NULL,
    DIST_RANGE STRING,
    AGE_RANGE STRING,
    IS_STRATEGIC_MBR INT NOT NULL,
    PREFERRED_CLUB_HAS_GAS INT NOT NULL,
    mbr_sid BIGINT NOT NULL,
    mbr_prmry_sid BIGINT NOT NULL,
    member_age DECIMAL(10,0),
    household_income STRING,
    MEMBER_FREQUENCY_GROUP STRING NOT NULL,
    `SPEND_IN_STORE_BY_TRIPS_LAST_TWENTY-SIX_WEEKS` DOUBLE NOT NULL,
    WEEK_OF_YEAR INT NOT NULL,
    MONTH INT NOT NULL,
    will_visit_from_0_2 INT NOT NULL,
    will_visit_from_3_5 INT NOT NULL,
    will_visit_from_5_7 INT NOT NULL,
    spend_from_0_2 DOUBLE NOT NULL,
    spend_from_3_5 DOUBLE NOT NULL,
    spend_from_5_7 DOUBLE NOT NULL,
    run_date DATE,
    run_name STRING,
    last_fiscal_week_training STRING,
    first_fiscal_week_training STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
# the column feature caused some issues when created using this statement. The approach that worked is to create the table directly from the df in the notebook itself.   We leave these cells in case in future refactores a fix for the issue is found. 

# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS   {model_trip_spend_etl_output} (
#     MBRSHP_SID BIGINT,
#     FISCAL_WEEK_END DATE,
#     FISCAL_WEEK_START DATE,
#     FISCAL_L4W_END DATE,
#     FISCAL_L8W_END DATE,
#     FISCAL_L12W_END DATE,
#     FISCAL_L26W_END DATE,
#     FISCAL_L52W_END DATE,
#     L52W_G4W_STDEV_TRIPS DOUBLE,
#     L52W_G4W_STDEV_SPEND DOUBLE,
#     WEEK_TRIPS BIGINT,
#     LAST_FOUR_WEEK_TRIPS BIGINT,
#     LAST_EIGHT_WEEK_TRIPS BIGINT,
#     LAST_TWELVE_WEEK_TRIPS BIGINT,
#     `LAST_TWENTY-SIX_WEEK_TRIPS` BIGINT,
#     `LAST_FIFTY-TWO_WEEK_TRIPS` BIGINT,
#     WEEK_SPEND DOUBLE,
#     FW_SPEND_IN_STORE DOUBLE,
#     WEEK_UNITS BIGINT,
#     WEEK_UNITS_OVER_FIFTY BIGINT,
#     FW_GAS_TRIPS BIGINT,
#     FW_GAS_DISTINCT_DAYS BIGINT,
#     FW_GAS_SPEND DOUBLE,
#     FW_GAS_AND_STORE_DISTINCT_DAYS BIGINT,
#     FW_ECOMMERCE_SPEND DOUBLE,
#     FW_ECOMMERCE_TRIPS BIGINT,
#     WEEK_DISTINCT_DAYS BIGINT,
#     LAST_FOUR_WEEK_DISTINCT_DAYS BIGINT,
#     LAST_EIGHT_WEEK_DISTINCT_DAYS BIGINT,
#     LAST_TWELVE_WEEK_DISTINCT_DAYS BIGINT,
#     `LAST_TWENTY-SIX_WEEK_DISTINCT_DAYS` BIGINT,
#     `LAST_FIFTY-TWO_WEEK_DISTINCT_DAYS` BIGINT,
#     WEEK_TRANSACTIONS BIGINT,
#     LAST_FOUR_WEEK_SPEND DOUBLE,
#     LAST_EIGHT_WEEK_SPEND DOUBLE,
#     LAST_TWELVE_WEEK_SPEND DOUBLE,
#     `LAST_TWENTY-SIX_WEEK_SPEND` DOUBLE,
#     `LAST_FIFTY-TWO_WEEK_SPEND` DOUBLE,
#     LFOURW_SPEND_IN_STORE DOUBLE,
#     LEIGHTW_SPEND_IN_STORE DOUBLE,
#     LTWELVEW_SPEND_IN_STORE DOUBLE,
#     `LTWENTY-SIXW_SPEND_IN_STORE` DOUBLE,
#     `LFIFTY-TWOW_SPEND_IN_STORE` DOUBLE,
#     LAST_FOUR_WEEK_UNITS BIGINT,
#     LAST_EIGHT_WEEK_UNITS BIGINT,
#     LAST_TWELVE_WEEK_UNITS BIGINT,
#     `LAST_TWENTY-SIX_WEEK_UNITS` BIGINT,
#     `LAST_FIFTY-TWO_WEEK_UNITS` BIGINT,
#     LAST_FOUR_WEEK_UNITS_OVER_FIFTY BIGINT,
#     LAST_EIGHT_WEEK_UNITS_OVER_FIFTY BIGINT,
#     LAST_TWELVE_WEEK_UNITS_OVER_FIFTY BIGINT,
#     `LAST_TWENTY-SIX_WEEK_UNITS_OVER_FIFTY` BIGINT,
#     `LAST_FIFTY-TWO_WEEK_UNITS_OVER_FIFTY` BIGINT,
#     L_FOURW_GAS_TRIPS BIGINT,
#     L_EIGHTW_GAS_TRIPS BIGINT,
#     L_TWELVEW_GAS_TRIPS BIGINT,
#     `L_TWENTY-SIXW_GAS_TRIPS` BIGINT,
#     `L_FIFTY-TWOW_GAS_TRIPS` BIGINT,
#     L_FOURW_GAS_DISTINCT_DAYS BIGINT,
#     L_EIGHTW_GAS_DISTINCT_DAYS BIGINT,
#     L_TWELVEW_GAS_DISTINCT_DAYS BIGINT,
#     `L_TWENTY-SIXW_GAS_DISTINCT_DAYS` BIGINT,
#     `L_FIFTY-TWOW_GAS_DISTINCT_DAYS` BIGINT,
#     L_FOURW_GAS_SPEND DOUBLE,
#     L_EIGHTW_GAS_SPEND DOUBLE,
#     L_TWELVEW_GAS_SPEND DOUBLE,
#     `L_TWENTY-SIXW_GAS_SPEND` DOUBLE,
#     `L_FIFTY-TWOW_GAS_SPEND` DOUBLE,
#     L_FOURW_GAS_AND_STORE_DISTINCT_DAYS BIGINT,
#     L_EIGHTW_GAS_AND_STORE_DISTINCT_DAYS BIGINT,
#     L_TWELVEW_GAS_AND_STORE_DISTINCT_DAYS BIGINT,
#     `L_TWENTY-SIXW_GAS_AND_STORE_DISTINCT_DAYS` BIGINT,
#     `L_FIFTY-TWOW_GAS_AND_STORE_DISTINCT_DAYS` BIGINT,
#     L_FOURW_ECOMMERCE_SPEND DOUBLE,
#     L_EIGHTW_ECOMMERCE_SPEND DOUBLE,
#     L_TWELVEW_ECOMMERCE_SPEND DOUBLE,
#     `L_TWENTY-SIXW_ECOMMERCE_SPEND` DOUBLE,
#     `L_FIFTY-TWOW_ECOMMERCE_SPEND` DOUBLE,
#     L_FOURW_ECOMMERCE_TRIPS BIGINT,
#     L_EIGHTW_ECOMMERCE_TRIPS BIGINT,
#     L_TWELVEW_ECOMMERCE_TRIPS BIGINT,
#     `L_TWENTY-SIXW_ECOMMERCE_TRIPS` BIGINT,
#     `L_FIFTY-TWOW_ECOMMERCE_TRIPS` BIGINT,
#     LAST_FOUR_WEEK_TRANSACTIONS BIGINT,
#     LAST_EIGHT_WEEK_TRANSACTIONS BIGINT,
#     LAST_TWELVE_WEEK_TRANSACTIONS BIGINT,
#     `LAST_TWENTY-SIX_WEEK_TRANSACTIONS` BIGINT,
#     `LAST_FIFTY-TWO_WEEK_TRANSACTIONS` BIGINT,
#     L52W_CREDIT DECIMAL(38,2),
#     L52W_DEBIT DECIMAL(38,2),
#     L52W_CASH DECIMAL(38,2),
#     L52W_COUPON DECIMAL(38,2),
#     L52W_EBT DECIMAL(38,2),
#     L52W_BJS DECIMAL(38,2),
#     FW_HAS_BOUGHT_MEN INT,
#     L52W_HAS_BOUGHT_MEN INT,
#     FW_HAS_BOUGHT_WOMEN INT,
#     L52W_HAS_BOUGHT_WOMEN INT,
#     FW_HAS_BOUGHT_PET INT,
#     L52W_HAS_BOUGHT_PET INT,
#     FW_HAS_BOUGHT_CHILDREN INT,
#     L52W_HAS_BOUGHT_CHILDREN INT,
#     FW_HAS_BOUGHT_BABY INT,
#     L52W_HAS_BOUGHT_BABY INT,
#     L52W_MEDIAN_BASKETSIZE FLOAT,
#     L52W_DISTINCT_CATEGORIES BIGINT,
#     LAST_FISCAL_WEEK_TRIP DATE,
#     LAST_TRIP DATE,
#     DAYS_SINCE_LAST_TRIP INT,
#     L52W_MAX_INTERVAL INT,
#     L52W_MIN_INTERVAL INT,
#     FW_ATC_CPN_RED BIGINT,
#     FW_ATC_CPN_SPEND DOUBLE,
#     L4W_ATC_CPN_RED BIGINT,
#     L4W_ATC_CPN_SPEND DOUBLE,
#     L8W_ATC_CPN_RED BIGINT,
#     L8W_ATC_CPN_SPEND DOUBLE,
#     L12W_ATC_CPN_RED BIGINT,
#     L12W_ATC_CPN_SPEND DOUBLE,
#     L26W_ATC_CPN_RED BIGINT,
#     L26W_ATC_CPN_SPEND DOUBLE,
#     L52W_ATC_CPN_RED BIGINT,
#     L52W_ATC_CPN_SPEND DOUBLE,
#     FW_CPN_CLP_COUNT BIGINT,
#     L4W_CPN_CLP_COUNT BIGINT,
#     L8W_CPN_CLP_COUNT BIGINT,
#     L12W_CPN_CLP_COUNT BIGINT,
#     L26W_CPN_CLP_COUNT BIGINT,
#     L52W_CPN_CLP_COUNT BIGINT,
#     FW_COUPON_REDEMPTIONS BIGINT,
#     FW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT,
#     FW_COUPON_SAVINGS DOUBLE,
#     FW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
#     LAST_FISCAL_WEEK_COUPON_REDEEMED DATE,
#     LAST_COUPON_REDEEMED DATE,
#     DAYS_SINCE_LAST_COUPON_REDEEMED INT,
#     LAST_FISCAL_WEEK_EMAIL_OPENED DATE,
#     LAST_EMAIL_OPENED DATE,
#     DAYS_SINCE_LAST_EMAIL_OPENED INT,
#     LAST_FISCAL_WEEK_ATC_CLIPPED DATE,
#     LAST_ATC_CLIPPED DATE,
#     DAYS_SINCE_LAST_ATC_CLIPPED INT,
#     EMAIL_OPEN_RATE DOUBLE,
#     FW_SAVINGS_W_CLPLSS DOUBLE,
#     cpn_channel STRING,
#     LFOURW_SAVINGS_W_CLPLSS DOUBLE,
#     LEIGHTW_SAVINGS_W_CLPLSS DOUBLE,
#     LTWELVEW_SAVINGS_W_CLPLSS DOUBLE,
#     `LTWENTY-SIXW_SAVINGS_W_CLPLSS` DOUBLE,
#     `LFIFTY-TWOW_SAVINGS_W_CLPLSS` DOUBLE,
#     MBERSHP_SID INT,
#     LFOURW_COUPON_REDEMPTIONS BIGINT,
#     LFOURW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT,
#     LFOURW_COUPON_SAVINGS DOUBLE,
#     LFOURW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
#     LEIGHTW_COUPON_REDEMPTIONS BIGINT,
#     LEIGHTW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT,
#     LEIGHTW_COUPON_SAVINGS DOUBLE,
#     LEIGHTW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
#     LTWELVEW_COUPON_REDEMPTIONS BIGINT,
#     LTWELVEW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT,
#     LTWELVEW_COUPON_SAVINGS DOUBLE,
#     LTWELVEW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
#     `LTWENTY-SIXW_COUPON_REDEMPTIONS` BIGINT,
#     `LTWENTY-SIXW_COUPON_REDEMPTIONS_W_CLPLSS` BIGINT,
#     `LTWENTY-SIXW_COUPON_SAVINGS` DOUBLE,
#     `LTWENTY-SIXW_COUPON_SAVINGS_W_CLPLSS` DOUBLE,
#     `LFIFTY-TWOW_COUPON_REDEMPTIONS` BIGINT,
#     `LFIFTY-TWOW_COUPON_REDEMPTIONS_W_CLPLSS` BIGINT,
#     `LFIFTY-TWOW_COUPON_SAVINGS` DOUBLE,
#     `LFIFTY-TWOW_COUPON_SAVINGS_W_CLPLSS` DOUBLE,
#     LATEST_MBRSHP_NBR STRING,
#     ZIP STRING,
#     EFF_DT DATE,
#     LATEST_MBRSHP_TYPE_ID STRING,
#     LATEST_MBRSHP_FEE_INC DECIMAL(5,2),
#     LATEST_MBRSHP_SUB_TYPE STRING,
#     LATEST_MBRSHP_ENR_DT DATE,
#     LATEST_MBRSHP_EXP_DT DATE,
#     LATEST_MBRSHP_RNWL_DT DATE,
#     LATEST_RWDS_MBR_IND STRING,
#     LATEST_MKT_CD STRING,
#     LATEST_HOME_ZIP_CD STRING,
#     LATEST_SIC_CD INT,
#     LATEST_CLUB_OF_FREQUENCY INT,
#     LATEST_PRI_SUPP_FHH_IND INT,
#     LATEST_GRP_AFFIL_ID STRING,
#     LATEST_ER_SIGNUP_DT DATE,
#     LATEST_AUTO_RNWL_IND STRING,
#     LATEST_TM_MBR_IND INT,
#     LATEST_TRIAL_MBR_IND INT,
#     LATEST_MFI_TIER INT,
#     EXP_DT DATE,
#     MBRSHP_STAT_CD STRING,
#     MBRSHP_EXP_DT DATE,
#     MBRSHP_RNWL_DT DATE,
#     MBRSHP_FEE_INC DOUBLE,
#     RWDS_MBR_IND STRING,
#     RWDS_MBR_ENR_DT DATE,
#     CLUB_OF_FREQUENCY INT,
#     TEAM_MBR_IND STRING,
#     FIRST_MBRSHP_FEE_INC DOUBLE,
#     ZIP_DISTANCE DOUBLE,
#     BJS_DRIVING_DISTANCE DOUBLE,
#     BJS_DISTANCE DOUBLE,
#     BJS_DRIVE_TIME DOUBLE,
#     WALMART_DRIVE_TIME DOUBLE,
#     WALMART_DRIVING_DISTANCE DOUBLE,
#     WALMART_DISTANCE DOUBLE,
#     COSTCO_DRIVE_TIME DOUBLE,
#     COSTCO_DRIVING_DISTANCE DOUBLE,
#     COSTCO_DISTANCE DOUBLE,
#     SAMS_DRIVE_TIME DOUBLE,
#     SAMS_DRIVING_DISTANCE DOUBLE,
#     SAMS_DISTANCE DOUBLE,
#     TENURE INT,
#     TENURE_GROUP STRING,
#     DAYS_UNTIL_EXP INT,
#     DAYS_SINCE_LAST_RNWL INT,
#     NUM_OF_RNWLS INT,
#     HAS_QUOTIENT_ID INT,
#     `L_FIFTY-TWOW_FIRST_MOST_SHOPPED_CATEGORY` STRING,
#     `L_FIFTY-TWOW_SECOND_MOST_SHOPPED_CATEGORY` STRING,
#     L52W_PREFERRED_CLUB_NBR INT,
#     L52W_PREFERRED_CLUB_TRIPS BIGINT,
#     L52W_PERCENT_TRIPS_PREFERRED_CLUB DOUBLE,
#     DUMMY_MBR INT,
#     L12W_SPEND_OVER_P12W_SPEND DOUBLE,
#     L26W_SPEND_OVER_P26W_SPEND DOUBLE,
#     L12W_TRIPS_OVER_P12W_TRIPS DOUBLE,
#     L26W_TRIPS_OVER_P26W_TRIPS DOUBLE,
#     STRATEGIC_MBR_HEADROOM DOUBLE,
#     DIST_RANGE STRING,
#     AGE_RANGE STRING,
#     IS_STRATEGIC_MBR INT,
#     PREFERRED_CLUB_HAS_GAS INT,
#     mbr_sid BIGINT,
#     mbr_prmry_sid BIGINT,
#     member_age DECIMAL(10,0),
#     household_income STRING,
#     MEMBER_FREQUENCY_GROUP STRING,
#     `SPEND_IN_STORE_BY_TRIPS_LAST_TWENTY-SIX_WEEKS` DOUBLE,
#     WEEK_OF_YEAR INT,
#     MONTH INT,
#     will_visit_from_0_2 INT,
#     will_visit_from_3_5 INT,
#     will_visit_from_5_7 INT,
#     spend_from_0_2 DOUBLE,
#     spend_from_3_5 DOUBLE,
#     spend_from_5_7 DOUBLE,
#     L52W_PREFERRED_CLUB_NBR_tmp DOUBLE,
#     FW_HAS_BOUGHT_MEN_tmp DOUBLE,
#     L52W_HAS_BOUGHT_MEN_tmp DOUBLE,
#     FW_HAS_BOUGHT_WOMEN_tmp DOUBLE,
#     L52W_HAS_BOUGHT_WOMEN_tmp DOUBLE,
#     FW_HAS_BOUGHT_PET_tmp DOUBLE,
#     L52W_HAS_BOUGHT_PET_tmp DOUBLE,
#     FW_HAS_BOUGHT_CHILDREN_tmp DOUBLE,
#     L52W_HAS_BOUGHT_CHILDREN_tmp DOUBLE,
#     FW_HAS_BOUGHT_BABY_tmp DOUBLE,
#     L52W_HAS_BOUGHT_BABY_tmp DOUBLE,
#     LATEST_MBRSHP_TYPE_ID_tmp DOUBLE,
#     LATEST_MBRSHP_FEE_INC_tmp DOUBLE,
#     LATEST_MBRSHP_SUB_TYPE_tmp DOUBLE,
#     LATEST_PRI_SUPP_FHH_IND_tmp DOUBLE,
#     LATEST_AUTO_RNWL_IND_tmp DOUBLE,
#     features STRUCT <type: TINYINT NOT NULL, size: INT, indices: ARRAY<INT>, values: ARRAY<DOUBLE>>
# )
# USING DELTA
# TBLPROPERTIES (
#     'predictiveOptimization'='true',
#     'delta.autoOptimize.optimizeWrite'='true',
#     'delta.autoOptimize.autoCompact'='true',
#     'delta.enableChangeDataFeed'='true',
#     'delta.columnMapping.mode'='name',
#     'delta.enableDeletionVectors' = 'true',
#     'delta.feature.deletionVectors' = 'supported',
#     'delta.feature.invariants' = 'supported'
# )
# """)

# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS   {model_trip_spend_etl_output_archive} (
#     MBRSHP_SID BIGINT,
#     FISCAL_WEEK_END DATE,
#     FISCAL_WEEK_START DATE,
#     FISCAL_L4W_END DATE,
#     FISCAL_L8W_END DATE,
#     FISCAL_L12W_END DATE,
#     FISCAL_L26W_END DATE,
#     FISCAL_L52W_END DATE,
#     L52W_G4W_STDEV_TRIPS DOUBLE,
#     L52W_G4W_STDEV_SPEND DOUBLE,
#     WEEK_TRIPS BIGINT,
#     LAST_FOUR_WEEK_TRIPS BIGINT,
#     LAST_EIGHT_WEEK_TRIPS BIGINT,
#     LAST_TWELVE_WEEK_TRIPS BIGINT,
#     `LAST_TWENTY-SIX_WEEK_TRIPS` BIGINT,
#     `LAST_FIFTY-TWO_WEEK_TRIPS` BIGINT,
#     WEEK_SPEND DOUBLE,
#     FW_SPEND_IN_STORE DOUBLE,
#     WEEK_UNITS BIGINT,
#     WEEK_UNITS_OVER_FIFTY BIGINT,
#     FW_GAS_TRIPS BIGINT,
#     FW_GAS_DISTINCT_DAYS BIGINT,
#     FW_GAS_SPEND DOUBLE,
#     FW_GAS_AND_STORE_DISTINCT_DAYS BIGINT,
#     FW_ECOMMERCE_SPEND DOUBLE,
#     FW_ECOMMERCE_TRIPS BIGINT,
#     WEEK_DISTINCT_DAYS BIGINT,
#     LAST_FOUR_WEEK_DISTINCT_DAYS BIGINT,
#     LAST_EIGHT_WEEK_DISTINCT_DAYS BIGINT,
#     LAST_TWELVE_WEEK_DISTINCT_DAYS BIGINT,
#     `LAST_TWENTY-SIX_WEEK_DISTINCT_DAYS` BIGINT,
#     `LAST_FIFTY-TWO_WEEK_DISTINCT_DAYS` BIGINT,
#     WEEK_TRANSACTIONS BIGINT,
#     LAST_FOUR_WEEK_SPEND DOUBLE,
#     LAST_EIGHT_WEEK_SPEND DOUBLE,
#     LAST_TWELVE_WEEK_SPEND DOUBLE,
#     `LAST_TWENTY-SIX_WEEK_SPEND` DOUBLE,
#     `LAST_FIFTY-TWO_WEEK_SPEND` DOUBLE,
#     LFOURW_SPEND_IN_STORE DOUBLE,
#     LEIGHTW_SPEND_IN_STORE DOUBLE,
#     LTWELVEW_SPEND_IN_STORE DOUBLE,
#     `LTWENTY-SIXW_SPEND_IN_STORE` DOUBLE,
#     `LFIFTY-TWOW_SPEND_IN_STORE` DOUBLE,
#     LAST_FOUR_WEEK_UNITS BIGINT,
#     LAST_EIGHT_WEEK_UNITS BIGINT,
#     LAST_TWELVE_WEEK_UNITS BIGINT,
#     `LAST_TWENTY-SIX_WEEK_UNITS` BIGINT,
#     `LAST_FIFTY-TWO_WEEK_UNITS` BIGINT,
#     LAST_FOUR_WEEK_UNITS_OVER_FIFTY BIGINT,
#     LAST_EIGHT_WEEK_UNITS_OVER_FIFTY BIGINT,
#     LAST_TWELVE_WEEK_UNITS_OVER_FIFTY BIGINT,
#     `LAST_TWENTY-SIX_WEEK_UNITS_OVER_FIFTY` BIGINT,
#     `LAST_FIFTY-TWO_WEEK_UNITS_OVER_FIFTY` BIGINT,
#     L_FOURW_GAS_TRIPS BIGINT,
#     L_EIGHTW_GAS_TRIPS BIGINT,
#     L_TWELVEW_GAS_TRIPS BIGINT,
#     `L_TWENTY-SIXW_GAS_TRIPS` BIGINT,
#     `L_FIFTY-TWOW_GAS_TRIPS` BIGINT,
#     L_FOURW_GAS_DISTINCT_DAYS BIGINT,
#     L_EIGHTW_GAS_DISTINCT_DAYS BIGINT,
#     L_TWELVEW_GAS_DISTINCT_DAYS BIGINT,
#     `L_TWENTY-SIXW_GAS_DISTINCT_DAYS` BIGINT,
#     `L_FIFTY-TWOW_GAS_DISTINCT_DAYS` BIGINT,
#     L_FOURW_GAS_SPEND DOUBLE,
#     L_EIGHTW_GAS_SPEND DOUBLE,
#     L_TWELVEW_GAS_SPEND DOUBLE,
#     `L_TWENTY-SIXW_GAS_SPEND` DOUBLE,
#     `L_FIFTY-TWOW_GAS_SPEND` DOUBLE,
#     L_FOURW_GAS_AND_STORE_DISTINCT_DAYS BIGINT,
#     L_EIGHTW_GAS_AND_STORE_DISTINCT_DAYS BIGINT,
#     L_TWELVEW_GAS_AND_STORE_DISTINCT_DAYS BIGINT,
#     `L_TWENTY-SIXW_GAS_AND_STORE_DISTINCT_DAYS` BIGINT,
#     `L_FIFTY-TWOW_GAS_AND_STORE_DISTINCT_DAYS` BIGINT,
#     L_FOURW_ECOMMERCE_SPEND DOUBLE,
#     L_EIGHTW_ECOMMERCE_SPEND DOUBLE,
#     L_TWELVEW_ECOMMERCE_SPEND DOUBLE,
#     `L_TWENTY-SIXW_ECOMMERCE_SPEND` DOUBLE,
#     `L_FIFTY-TWOW_ECOMMERCE_SPEND` DOUBLE,
#     L_FOURW_ECOMMERCE_TRIPS BIGINT,
#     L_EIGHTW_ECOMMERCE_TRIPS BIGINT,
#     L_TWELVEW_ECOMMERCE_TRIPS BIGINT,
#     `L_TWENTY-SIXW_ECOMMERCE_TRIPS` BIGINT,
#     `L_FIFTY-TWOW_ECOMMERCE_TRIPS` BIGINT,
#     LAST_FOUR_WEEK_TRANSACTIONS BIGINT,
#     LAST_EIGHT_WEEK_TRANSACTIONS BIGINT,
#     LAST_TWELVE_WEEK_TRANSACTIONS BIGINT,
#     `LAST_TWENTY-SIX_WEEK_TRANSACTIONS` BIGINT,
#     `LAST_FIFTY-TWO_WEEK_TRANSACTIONS` BIGINT,
#     L52W_CREDIT DECIMAL(38,2),
#     L52W_DEBIT DECIMAL(38,2),
#     L52W_CASH DECIMAL(38,2),
#     L52W_COUPON DECIMAL(38,2),
#     L52W_EBT DECIMAL(38,2),
#     L52W_BJS DECIMAL(38,2),
#     FW_HAS_BOUGHT_MEN INT,
#     L52W_HAS_BOUGHT_MEN INT,
#     FW_HAS_BOUGHT_WOMEN INT,
#     L52W_HAS_BOUGHT_WOMEN INT,
#     FW_HAS_BOUGHT_PET INT,
#     L52W_HAS_BOUGHT_PET INT,
#     FW_HAS_BOUGHT_CHILDREN INT,
#     L52W_HAS_BOUGHT_CHILDREN INT,
#     FW_HAS_BOUGHT_BABY INT,
#     L52W_HAS_BOUGHT_BABY INT,
#     L52W_MEDIAN_BASKETSIZE FLOAT,
#     L52W_DISTINCT_CATEGORIES BIGINT,
#     LAST_FISCAL_WEEK_TRIP DATE,
#     LAST_TRIP DATE,
#     DAYS_SINCE_LAST_TRIP INT,
#     L52W_MAX_INTERVAL INT,
#     L52W_MIN_INTERVAL INT,
#     FW_ATC_CPN_RED BIGINT,
#     FW_ATC_CPN_SPEND DOUBLE,
#     L4W_ATC_CPN_RED BIGINT,
#     L4W_ATC_CPN_SPEND DOUBLE,
#     L8W_ATC_CPN_RED BIGINT,
#     L8W_ATC_CPN_SPEND DOUBLE,
#     L12W_ATC_CPN_RED BIGINT,
#     L12W_ATC_CPN_SPEND DOUBLE,
#     L26W_ATC_CPN_RED BIGINT,
#     L26W_ATC_CPN_SPEND DOUBLE,
#     L52W_ATC_CPN_RED BIGINT,
#     L52W_ATC_CPN_SPEND DOUBLE,
#     FW_CPN_CLP_COUNT BIGINT,
#     L4W_CPN_CLP_COUNT BIGINT,
#     L8W_CPN_CLP_COUNT BIGINT,
#     L12W_CPN_CLP_COUNT BIGINT,
#     L26W_CPN_CLP_COUNT BIGINT,
#     L52W_CPN_CLP_COUNT BIGINT,
#     FW_COUPON_REDEMPTIONS BIGINT,
#     FW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT,
#     FW_COUPON_SAVINGS DOUBLE,
#     FW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
#     LAST_FISCAL_WEEK_COUPON_REDEEMED DATE,
#     LAST_COUPON_REDEEMED DATE,
#     DAYS_SINCE_LAST_COUPON_REDEEMED INT,
#     LAST_FISCAL_WEEK_EMAIL_OPENED DATE,
#     LAST_EMAIL_OPENED DATE,
#     DAYS_SINCE_LAST_EMAIL_OPENED INT,
#     LAST_FISCAL_WEEK_ATC_CLIPPED DATE,
#     LAST_ATC_CLIPPED DATE,
#     DAYS_SINCE_LAST_ATC_CLIPPED INT,
#     EMAIL_OPEN_RATE DOUBLE,
#     FW_SAVINGS_W_CLPLSS DOUBLE,
#     cpn_channel STRING,
#     LFOURW_SAVINGS_W_CLPLSS DOUBLE,
#     LEIGHTW_SAVINGS_W_CLPLSS DOUBLE,
#     LTWELVEW_SAVINGS_W_CLPLSS DOUBLE,
#     `LTWENTY-SIXW_SAVINGS_W_CLPLSS` DOUBLE,
#     `LFIFTY-TWOW_SAVINGS_W_CLPLSS` DOUBLE,
#     MBERSHP_SID INT,
#     LFOURW_COUPON_REDEMPTIONS BIGINT,
#     LFOURW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT,
#     LFOURW_COUPON_SAVINGS DOUBLE,
#     LFOURW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
#     LEIGHTW_COUPON_REDEMPTIONS BIGINT,
#     LEIGHTW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT,
#     LEIGHTW_COUPON_SAVINGS DOUBLE,
#     LEIGHTW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
#     LTWELVEW_COUPON_REDEMPTIONS BIGINT,
#     LTWELVEW_COUPON_REDEMPTIONS_W_CLPLSS BIGINT,
#     LTWELVEW_COUPON_SAVINGS DOUBLE,
#     LTWELVEW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
#     `LTWENTY-SIXW_COUPON_REDEMPTIONS` BIGINT,
#     `LTWENTY-SIXW_COUPON_REDEMPTIONS_W_CLPLSS` BIGINT,
#     `LTWENTY-SIXW_COUPON_SAVINGS` DOUBLE,
#     `LTWENTY-SIXW_COUPON_SAVINGS_W_CLPLSS` DOUBLE,
#     `LFIFTY-TWOW_COUPON_REDEMPTIONS` BIGINT,
#     `LFIFTY-TWOW_COUPON_REDEMPTIONS_W_CLPLSS` BIGINT,
#     `LFIFTY-TWOW_COUPON_SAVINGS` DOUBLE,
#     `LFIFTY-TWOW_COUPON_SAVINGS_W_CLPLSS` DOUBLE,
#     LATEST_MBRSHP_NBR STRING,
#     ZIP STRING,
#     EFF_DT DATE,
#     LATEST_MBRSHP_TYPE_ID STRING,
#     LATEST_MBRSHP_FEE_INC DECIMAL(5,2),
#     LATEST_MBRSHP_SUB_TYPE STRING,
#     LATEST_MBRSHP_ENR_DT DATE,
#     LATEST_MBRSHP_EXP_DT DATE,
#     LATEST_MBRSHP_RNWL_DT DATE,
#     LATEST_RWDS_MBR_IND STRING,
#     LATEST_MKT_CD STRING,
#     LATEST_HOME_ZIP_CD STRING,
#     LATEST_SIC_CD INT,
#     LATEST_CLUB_OF_FREQUENCY INT,
#     LATEST_PRI_SUPP_FHH_IND INT,
#     LATEST_GRP_AFFIL_ID STRING,
#     LATEST_ER_SIGNUP_DT DATE,
#     LATEST_AUTO_RNWL_IND STRING,
#     LATEST_TM_MBR_IND INT,
#     LATEST_TRIAL_MBR_IND INT,
#     LATEST_MFI_TIER INT,
#     EXP_DT DATE,
#     MBRSHP_STAT_CD STRING,
#     MBRSHP_EXP_DT DATE,
#     MBRSHP_RNWL_DT DATE,
#     MBRSHP_FEE_INC DOUBLE,
#     RWDS_MBR_IND STRING,
#     RWDS_MBR_ENR_DT DATE,
#     CLUB_OF_FREQUENCY INT,
#     TEAM_MBR_IND STRING,
#     FIRST_MBRSHP_FEE_INC DOUBLE,
#     ZIP_DISTANCE DOUBLE,
#     BJS_DRIVING_DISTANCE DOUBLE,
#     BJS_DISTANCE DOUBLE,
#     BJS_DRIVE_TIME DOUBLE,
#     WALMART_DRIVE_TIME DOUBLE,
#     WALMART_DRIVING_DISTANCE DOUBLE,
#     WALMART_DISTANCE DOUBLE,
#     COSTCO_DRIVE_TIME DOUBLE,
#     COSTCO_DRIVING_DISTANCE DOUBLE,
#     COSTCO_DISTANCE DOUBLE,
#     SAMS_DRIVE_TIME DOUBLE,
#     SAMS_DRIVING_DISTANCE DOUBLE,
#     SAMS_DISTANCE DOUBLE,
#     TENURE INT,
#     TENURE_GROUP STRING,
#     DAYS_UNTIL_EXP INT,
#     DAYS_SINCE_LAST_RNWL INT,
#     NUM_OF_RNWLS INT,
#     HAS_QUOTIENT_ID INT,
#     `L_FIFTY-TWOW_FIRST_MOST_SHOPPED_CATEGORY` STRING,
#     `L_FIFTY-TWOW_SECOND_MOST_SHOPPED_CATEGORY` STRING,
#     L52W_PREFERRED_CLUB_NBR INT,
#     L52W_PREFERRED_CLUB_TRIPS BIGINT,
#     L52W_PERCENT_TRIPS_PREFERRED_CLUB DOUBLE,
#     DUMMY_MBR INT,
#     L12W_SPEND_OVER_P12W_SPEND DOUBLE,
#     L26W_SPEND_OVER_P26W_SPEND DOUBLE,
#     L12W_TRIPS_OVER_P12W_TRIPS DOUBLE,
#     L26W_TRIPS_OVER_P26W_TRIPS DOUBLE,
#     STRATEGIC_MBR_HEADROOM DOUBLE,
#     DIST_RANGE STRING,
#     AGE_RANGE STRING,
#     IS_STRATEGIC_MBR INT,
#     PREFERRED_CLUB_HAS_GAS INT,
#     mbr_sid BIGINT,
#     mbr_prmry_sid BIGINT,
#     member_age DECIMAL(10,0),
#     household_income STRING,
#     MEMBER_FREQUENCY_GROUP STRING,
#     `SPEND_IN_STORE_BY_TRIPS_LAST_TWENTY-SIX_WEEKS` DOUBLE,
#     WEEK_OF_YEAR INT,
#     MONTH INT,
#     will_visit_from_0_2 INT,
#     will_visit_from_3_5 INT,
#     will_visit_from_5_7 INT,
#     spend_from_0_2 DOUBLE,
#     spend_from_3_5 DOUBLE,
#     spend_from_5_7 DOUBLE,
#     L52W_PREFERRED_CLUB_NBR_tmp DOUBLE,
#     FW_HAS_BOUGHT_MEN_tmp DOUBLE,
#     L52W_HAS_BOUGHT_MEN_tmp DOUBLE,
#     FW_HAS_BOUGHT_WOMEN_tmp DOUBLE,
#     L52W_HAS_BOUGHT_WOMEN_tmp DOUBLE,
#     FW_HAS_BOUGHT_PET_tmp DOUBLE,
#     L52W_HAS_BOUGHT_PET_tmp DOUBLE,
#     FW_HAS_BOUGHT_CHILDREN_tmp DOUBLE,
#     L52W_HAS_BOUGHT_CHILDREN_tmp DOUBLE,
#     FW_HAS_BOUGHT_BABY_tmp DOUBLE,
#     L52W_HAS_BOUGHT_BABY_tmp DOUBLE,
#     LATEST_MBRSHP_TYPE_ID_tmp DOUBLE,
#     LATEST_MBRSHP_FEE_INC_tmp DOUBLE,
#     LATEST_MBRSHP_SUB_TYPE_tmp DOUBLE,
#     LATEST_PRI_SUPP_FHH_IND_tmp DOUBLE,
#     LATEST_AUTO_RNWL_IND_tmp DOUBLE,
#     features STRUCT <type: TINYINT NOT NULL, size: INT, indices: ARRAY<INT>, values: ARRAY<DOUBLE>>,
#     run_date DATE,
#     run_name STRING,
#     last_fiscal_week_training STRING,
#     first_fiscal_week_training STRING
# )
# USING DELTA
# TBLPROPERTIES (
#     'predictiveOptimization'='true',
#     'delta.autoOptimize.optimizeWrite'='true',
#     'delta.autoOptimize.autoCompact'='true',
#     'delta.enableChangeDataFeed'='true',
#     'delta.columnMapping.mode'='name',
#     'delta.enableDeletionVectors' = 'true',
#     'delta.feature.deletionVectors' = 'supported',
#     'delta.feature.invariants' = 'supported'
# )
# """)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {trip_spend_prediction} (
    MBRSHP_SID BIGINT,
    LATEST_PRI_SUPP_FHH_IND INT,
    probability_making_a_trip DOUBLE,
    predicted_make_trip INT,
    predicted_spend DOUBLE,
    FISCAL_WEEK_END DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {cf_data_by_cat} (
    CATEGORY_ID LONG,
    TRIPS BIGINT,
    SALES_AMT DOUBLE,
    SALES_UNITS BIGINT,
    CATEGORY_CD STRING,
    START_DATE DATE,
    END_DATE DATE,
    RUN_NAME DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {cf_matrix} (
    MBRSHP_SID BIGINT,
    CATEGORY_ID LONG,
    TRIPS BIGINT,
    SALES_AMT DOUBLE,
    SALES_UNITS BIGINT,
    CATEGORY_CD STRING,
    START_DATE DATE,
    END_DATE DATE,
    RUN_NAME DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {cf_cat_lookup} (
    CATEGORY_NAME STRING,
    CATEGORY_ID LONG,
    CATEGORY_CD STRING,
    START_DATE DATE,
    END_DATE DATE,
    RUN_NAME DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {cf_slate} (
    MBRSHP_SID LONG,
    CATEGORY_ID LONG,
    CATEGORY_CD STRING,
    START_DATE DATE,
    END_DATE DATE,
    RUN_NAME DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {cf_prediction} (
    CATEGORY_ID INT,
    MBRSHP_SID INT,
    prediction FLOAT,
    CATEGORY_NAME STRING,
    CATEGORY_LVL STRING,
    hs_ind_lambda5 STRING,
    hs_ind_lambdav2_5 STRING,
    hs_ind_lambda15 STRING,
    hs_ind_lambdav2_15 STRING,
    hs_ind_lambda30 STRING,
    hs_ind_lambdav2_30 STRING,
    hs_ind_lambda50 STRING,
    hs_ind_lambdav2_50 STRING,
    hs_ind_lambda80 STRING,
    hs_ind_lambdav2_80 STRING,
    prediction_v2 DOUBLE,
    CATEGORY_CD STRING,
    START_DATE DATE,
    END_DATE DATE,
    RUN_NAME DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {cf_invalid} (
    summary STRING,
    TENURE STRING,
    LAST_EIGHT_WEEK_SPEND STRING,
    LAST_TWELVE_WEEK_SPEND STRING,
    `LAST_TWENTY-SIX_WEEK_SPEND` STRING,
    `LAST_FIFTY-TWO_WEEK_SPEND` STRING,
    WEEK_TRIPS STRING,
    LAST_FOUR_WEEK_TRIPS STRING,
    LAST_EIGHT_WEEK_TRIPS STRING,
    LAST_TWELVE_WEEK_TRIPS STRING,
    `LAST_TWENTY-SIX_WEEK_TRIPS` STRING,
    `LAST_FIFTY-TWO_WEEK_TRIPS` STRING,
    RUN_NAME DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {cf_top_cat} (
    CATEGORY_ID INT,
    CATEGORY_NAME STRING,
    hook_stretch STRING,
    mbrs_count BIGINT,
    cat_rank INT,
    lambda STRING,
    RUN_NAME DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gm_etl_output} (
    MBRSHP_SID LONG,
    GM_purchase LONG,
    FISCAL_WEEK_END DATE,
    FISCAL_WEEK_START DATE,
    FISCAL_L4W_END DATE,
    FISCAL_L8W_END DATE,
    FISCAL_L12W_END DATE,
    FISCAL_L26W_END DATE,
    FISCAL_L52W_END DATE,
    L52W_G4W_STDEV_TRIPS DOUBLE,
    L52W_G4W_STDEV_SPEND DOUBLE,
    WEEK_TRIPS LONG,
    LAST_FOUR_WEEK_TRIPS LONG,
    LAST_EIGHT_WEEK_TRIPS LONG,
    LAST_TWELVE_WEEK_TRIPS LONG,
    `LAST_TWENTY-SIX_WEEK_TRIPS` LONG,
    `LAST_FIFTY-TWO_WEEK_TRIPS` LONG,
    WEEK_SPEND DOUBLE,
    FW_SPEND_IN_STORE DOUBLE,
    WEEK_UNITS LONG,
    WEEK_UNITS_OVER_FIFTY LONG,
    FW_GAS_TRIPS LONG,
    FW_GAS_DISTINCT_DAYS LONG,
    FW_GAS_SPEND DOUBLE,
    FW_GAS_AND_STORE_DISTINCT_DAYS LONG,
    FW_ECOMMERCE_SPEND DOUBLE,
    FW_ECOMMERCE_TRIPS LONG,
    WEEK_DISTINCT_DAYS LONG,
    LAST_FOUR_WEEK_DISTINCT_DAYS LONG,
    LAST_EIGHT_WEEK_DISTINCT_DAYS LONG,
    LAST_TWELVE_WEEK_DISTINCT_DAYS LONG,
    `LAST_TWENTY-SIX_WEEK_DISTINCT_DAYS` LONG,
    `LAST_FIFTY-TWO_WEEK_DISTINCT_DAYS` LONG,
    WEEK_TRANSACTIONS LONG,
    LAST_FOUR_WEEK_SPEND DOUBLE,
    LAST_EIGHT_WEEK_SPEND DOUBLE,
    LAST_TWELVE_WEEK_SPEND DOUBLE,
    `LAST_TWENTY-SIX_WEEK_SPEND` DOUBLE,
    `LAST_FIFTY-TWO_WEEK_SPEND` DOUBLE,
    LFOURW_SPEND_IN_STORE DOUBLE,
    LEIGHTW_SPEND_IN_STORE DOUBLE,
    LTWELVEW_SPEND_IN_STORE DOUBLE,
    `LTWENTY-SIXW_SPEND_IN_STORE` DOUBLE,
    `LFIFTY-TWOW_SPEND_IN_STORE` DOUBLE,
    LAST_FOUR_WEEK_UNITS LONG,
    LAST_EIGHT_WEEK_UNITS LONG,
    LAST_TWELVE_WEEK_UNITS LONG,
    `LAST_TWENTY-SIX_WEEK_UNITS` LONG,
    `LAST_FIFTY-TWO_WEEK_UNITS` LONG,
    LAST_FOUR_WEEK_UNITS_OVER_FIFTY LONG,
    LAST_EIGHT_WEEK_UNITS_OVER_FIFTY LONG,
    LAST_TWELVE_WEEK_UNITS_OVER_FIFTY LONG,
    `LAST_TWENTY-SIX_WEEK_UNITS_OVER_FIFTY` LONG,
    `LAST_FIFTY-TWO_WEEK_UNITS_OVER_FIFTY` LONG,
    L_FOURW_GAS_TRIPS LONG,
    L_EIGHTW_GAS_TRIPS LONG,
    L_TWELVEW_GAS_TRIPS LONG,
    `L_TWENTY-SIXW_GAS_TRIPS` LONG,
    `L_FIFTY-TWOW_GAS_TRIPS` LONG,
    L_FOURW_GAS_DISTINCT_DAYS LONG,
    L_EIGHTW_GAS_DISTINCT_DAYS LONG,
    L_TWELVEW_GAS_DISTINCT_DAYS LONG,
    `L_TWENTY-SIXW_GAS_DISTINCT_DAYS` LONG,
    `L_FIFTY-TWOW_GAS_DISTINCT_DAYS` LONG,
    L_FOURW_GAS_SPEND DOUBLE,
    L_EIGHTW_GAS_SPEND DOUBLE,
    L_TWELVEW_GAS_SPEND DOUBLE,
    `L_TWENTY-SIXW_GAS_SPEND` DOUBLE,
    `L_FIFTY-TWOW_GAS_SPEND` DOUBLE,
    L_FOURW_GAS_AND_STORE_DISTINCT_DAYS LONG,
    L_EIGHTW_GAS_AND_STORE_DISTINCT_DAYS LONG,
    L_TWELVEW_GAS_AND_STORE_DISTINCT_DAYS LONG,
    `L_TWENTY-SIXW_GAS_AND_STORE_DISTINCT_DAYS` LONG,
    `L_FIFTY-TWOW_GAS_AND_STORE_DISTINCT_DAYS` LONG,
    L_FOURW_ECOMMERCE_SPEND DOUBLE,
    L_EIGHTW_ECOMMERCE_SPEND DOUBLE,
    L_TWELVEW_ECOMMERCE_SPEND DOUBLE,
    `L_TWENTY-SIXW_ECOMMERCE_SPEND` DOUBLE,
    `L_FIFTY-TWOW_ECOMMERCE_SPEND` DOUBLE,
    L_FOURW_ECOMMERCE_TRIPS LONG,
    L_EIGHTW_ECOMMERCE_TRIPS LONG,
    L_TWELVEW_ECOMMERCE_TRIPS LONG,
    `L_TWENTY-SIXW_ECOMMERCE_TRIPS` LONG,
    `L_FIFTY-TWOW_ECOMMERCE_TRIPS` LONG,
    LAST_FOUR_WEEK_TRANSACTIONS LONG,
    LAST_EIGHT_WEEK_TRANSACTIONS LONG,
    LAST_TWELVE_WEEK_TRANSACTIONS LONG,
    `LAST_TWENTY-SIX_WEEK_TRANSACTIONS` LONG,
    `LAST_FIFTY-TWO_WEEK_TRANSACTIONS` LONG,
    L52W_CREDIT DECIMAL(38,2),
    L52W_DEBIT DECIMAL(38,2),
    L52W_CASH DECIMAL(38,2),
    L52W_COUPON DECIMAL(38,2),
    L52W_EBT DECIMAL(38,2),
    L52W_BJS DECIMAL(38,2),
    FW_HAS_BOUGHT_MEN INTEGER,
    L52W_HAS_BOUGHT_MEN INTEGER,
    FW_HAS_BOUGHT_WOMEN INTEGER,
    L52W_HAS_BOUGHT_WOMEN INTEGER,
    FW_HAS_BOUGHT_PET INTEGER,
    L52W_HAS_BOUGHT_PET INTEGER,
    FW_HAS_BOUGHT_CHILDREN INTEGER,
    L52W_HAS_BOUGHT_CHILDREN INTEGER,
    FW_HAS_BOUGHT_BABY INTEGER,
    L52W_HAS_BOUGHT_BABY INTEGER,
    L52W_MEDIAN_BASKETSIZE FLOAT,
    L52W_DISTINCT_CATEGORIES LONG,
    LAST_FISCAL_WEEK_TRIP DATE,
    LAST_TRIP DATE,
    DAYS_SINCE_LAST_TRIP INTEGER,
    L52W_MAX_INTERVAL INTEGER,
    L52W_MIN_INTERVAL INTEGER,
    FW_ATC_CPN_RED LONG,
    FW_ATC_CPN_SPEND DOUBLE,
    L4W_ATC_CPN_RED LONG,
    L4W_ATC_CPN_SPEND DOUBLE,
    L8W_ATC_CPN_RED LONG,
    L8W_ATC_CPN_SPEND DOUBLE,
    L12W_ATC_CPN_RED LONG,
    L12W_ATC_CPN_SPEND DOUBLE,
    L26W_ATC_CPN_RED LONG,
    L26W_ATC_CPN_SPEND DOUBLE,
    L52W_ATC_CPN_RED LONG,
    L52W_ATC_CPN_SPEND DOUBLE,
    FW_CPN_CLP_COUNT LONG,
    L4W_CPN_CLP_COUNT LONG,
    L8W_CPN_CLP_COUNT LONG,
    L12W_CPN_CLP_COUNT LONG,
    L26W_CPN_CLP_COUNT LONG,
    L52W_CPN_CLP_COUNT LONG,
    FW_COUPON_REDEMPTIONS LONG,
    FW_COUPON_REDEMPTIONS_W_CLPLSS LONG,
    FW_COUPON_SAVINGS DOUBLE,
    FW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
    LAST_FISCAL_WEEK_COUPON_REDEEMED DATE,
    LAST_COUPON_REDEEMED DATE,
    DAYS_SINCE_LAST_COUPON_REDEEMED INTEGER,
    LAST_FISCAL_WEEK_EMAIL_OPENED DATE,
    LAST_EMAIL_OPENED DATE,
    DAYS_SINCE_LAST_EMAIL_OPENED INTEGER,
    LAST_FISCAL_WEEK_ATC_CLIPPED DATE,
    LAST_ATC_CLIPPED DATE,
    DAYS_SINCE_LAST_ATC_CLIPPED INTEGER,
    EMAIL_OPEN_RATE DOUBLE,
    FW_SAVINGS_W_CLPLSS DOUBLE,
    cpn_channel STRING,
    LFOURW_SAVINGS_W_CLPLSS DOUBLE,
    LEIGHTW_SAVINGS_W_CLPLSS DOUBLE,
    LTWELVEW_SAVINGS_W_CLPLSS DOUBLE,
    `LTWENTY-SIXW_SAVINGS_W_CLPLSS` DOUBLE,
    `LFIFTY-TWOW_SAVINGS_W_CLPLSS` DOUBLE,
    LFOURW_COUPON_REDEMPTIONS LONG,
    LFOURW_COUPON_REDEMPTIONS_W_CLPLSS LONG,
    LFOURW_COUPON_SAVINGS DOUBLE,
    LFOURW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
    LEIGHTW_COUPON_REDEMPTIONS LONG,
    LEIGHTW_COUPON_REDEMPTIONS_W_CLPLSS LONG,
    LEIGHTW_COUPON_SAVINGS DOUBLE,
    LEIGHTW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
    LTWELVEW_COUPON_REDEMPTIONS LONG,
    LTWELVEW_COUPON_REDEMPTIONS_W_CLPLSS LONG,
    LTWELVEW_COUPON_SAVINGS DOUBLE,
    LTWELVEW_COUPON_SAVINGS_W_CLPLSS DOUBLE,
    `LTWENTY-SIXW_COUPON_REDEMPTIONS` LONG,
    `LTWENTY-SIXW_COUPON_REDEMPTIONS_W_CLPLSS` LONG,
    `LTWENTY-SIXW_COUPON_SAVINGS` DOUBLE,
    `LTWENTY-SIXW_COUPON_SAVINGS_W_CLPLSS` DOUBLE,
    `LFIFTY-TWOW_COUPON_REDEMPTIONS` LONG,
    `LFIFTY-TWOW_COUPON_REDEMPTIONS_W_CLPLSS` LONG,
    `LFIFTY-TWOW_COUPON_SAVINGS` DOUBLE,
    `LFIFTY-TWOW_COUPON_SAVINGS_W_CLPLSS` DOUBLE,
    LATEST_MBRSHP_NBR STRING,
    ZIP STRING,
    EFF_DT DATE,
    LATEST_MBRSHP_TYPE_ID STRING,
    LATEST_MBRSHP_FEE_INC DECIMAL(5,2),
    LATEST_MBRSHP_SUB_TYPE STRING,
    LATEST_MBRSHP_ENR_DT DATE,
    LATEST_MBRSHP_EXP_DT DATE,
    LATEST_MBRSHP_RNWL_DT DATE,
    LATEST_RWDS_MBR_IND STRING,
    LATEST_MKT_CD STRING,
    LATEST_HOME_ZIP_CD STRING,
    LATEST_SIC_CD INTEGER,
    LATEST_CLUB_OF_FREQUENCY INTEGER,
    LATEST_PRI_SUPP_FHH_IND INTEGER,
    LATEST_GRP_AFFIL_ID STRING,
    LATEST_ER_SIGNUP_DT DATE,
    LATEST_AUTO_RNWL_IND STRING,
    LATEST_TM_MBR_IND INTEGER,
    LATEST_TRIAL_MBR_IND INTEGER,
    LATEST_MFI_TIER INTEGER,
    EXP_DT DATE,
    MBRSHP_STAT_CD STRING,
    MBRSHP_EXP_DT DATE,
    MBRSHP_RNWL_DT DATE,
    MBRSHP_FEE_INC DOUBLE,
    RWDS_MBR_IND STRING,
    RWDS_MBR_ENR_DT DATE,
    CLUB_OF_FREQUENCY INTEGER,
    TEAM_MBR_IND STRING,
    FIRST_MBRSHP_FEE_INC DOUBLE,
    ZIP_DISTANCE DOUBLE,
    BJS_DRIVING_DISTANCE DOUBLE,
    BJS_DISTANCE DOUBLE,
    BJS_DRIVE_TIME DOUBLE,
    WALMART_DRIVE_TIME DOUBLE,
    WALMART_DRIVING_DISTANCE DOUBLE,
    WALMART_DISTANCE DOUBLE,
    COSTCO_DRIVE_TIME DOUBLE,
    COSTCO_DRIVING_DISTANCE DOUBLE,
    COSTCO_DISTANCE DOUBLE,
    SAMS_DRIVE_TIME DOUBLE,
    SAMS_DRIVING_DISTANCE DOUBLE,
    SAMS_DISTANCE DOUBLE,
    TENURE INTEGER,
    TENURE_GROUP STRING,
    DAYS_UNTIL_EXP INTEGER,
    DAYS_SINCE_LAST_RNWL INTEGER,
    NUM_OF_RNWLS INTEGER,
    HAS_QUOTIENT_ID INTEGER,
    `L_FIFTY-TWOW_FIRST_MOST_SHOPPED_CATEGORY` STRING,
    `L_FIFTY-TWOW_SECOND_MOST_SHOPPED_CATEGORY` STRING,
    L52W_PREFERRED_CLUB_NBR INTEGER,
    L52W_PREFERRED_CLUB_TRIPS LONG,
    L52W_PERCENT_TRIPS_PREFERRED_CLUB DOUBLE,
    DUMMY_MBR INTEGER,
    L12W_SPEND_OVER_P12W_SPEND DOUBLE,
    L26W_SPEND_OVER_P26W_SPEND DOUBLE,
    L12W_TRIPS_OVER_P12W_TRIPS DOUBLE,
    L26W_TRIPS_OVER_P26W_TRIPS DOUBLE,
    STRATEGIC_MBR_HEADROOM DOUBLE,
    DIST_RANGE STRING,
    AGE_RANGE STRING,
    IS_STRATEGIC_MBR INTEGER,
    PREFERRED_CLUB_HAS_GAS INTEGER,
    mbr_sid LONG,
    mbr_prmry_sid LONG,
    member_age DECIMAL(10,0),
    household_income STRING,
    MEMBER_FREQUENCY_GROUP STRING,
    Shopped_in_Last_3Month INTEGER,
    Sundries_trips_last_1y LONG,
    unique_sundries_sub_cat_last_1y LONG,
    Sundries_sales_last_1y DOUBLE,
    Sundries_trips_last_2y LONG,
    unique_sundries_sub_cat_last_2y LONG,
    Sundries_sales_last_2y DOUBLE,
    Sundries_trips_last_3y LONG,
    unique_sundries_sub_cat_last_3y LONG,
    Sundries_sales_last_3y DOUBLE,
    Perishables_trips_last_1y LONG,
    unique_Perishables_sub_cat_last_1y LONG,
    Perishables_sales_last_1y DOUBLE,
    Perishables_trips_last_2y LONG,
    unique_Perishables_sub_cat_last_2y LONG,
    Perishables_sales_last_2y DOUBLE,
    Perishables_trips_last_3y LONG,
    unique_Perishables_sub_cat_last_3y LONG,
    Perishables_sales_last_3y DOUBLE,
    Grocery_trips_last_1y LONG,
    unique_Grocery_sub_cat_last_1y LONG,
    Grocery_sales_last_1y DOUBLE,
    Grocery_trips_last_2y LONG,
    unique_Grocery_sub_cat_last_2y LONG,
    Grocery_sales_last_2y DOUBLE,
    Grocery_trips_last_3y LONG,
    unique_Grocery_sub_cat_last_3y LONG,
    Grocery_sales_last_3y DOUBLE,
    GM_trips_last_1y LONG,
    unique_GM_sub_cat_last_1y LONG,
    GM_sales_last_1y DOUBLE,
    GM_trips_last_2y LONG,
    unique_GM_sub_cat_last_2y LONG,
    GM_sales_last_2y DOUBLE,
    GM_trips_last_3y LONG,
    unique_GM_sub_cat_last_3y LONG,
    GM_sales_last_3y DOUBLE,
    BEVERAGES LONG,
    CONSUMABLES LONG,
    CROSS_MERCHANDISE LONG,
    ELCTRNCS___ENTRTNMNT LONG,
    FASHION LONG,
    FOOD_SERVICES LONG,
    FRESH LONG,
    FROZEN LONG,
    GEN_MERCH_ROADSHOWS LONG,
    GROCERY_ROADSHOW LONG,
    HOME___OFFICE LONG,
    PANTRY_GOODS LONG,
    PERISHABLE_ROADSHOW LONG,
    PERSONAL_CARE LONG,
    PET LONG,
    SPECIALTY_BUSINESS LONG,
    SUNDRIES_ROADSHOW LONG,
    TOBACCO LONG,
    UNKNOWN LONG,
    redeem_paper_cpn_last_1y LONG,
    redeem_paper_cpn_last_2y LONG,
    redeem_paper_cpn_last_3y LONG,
    redeem_digital_cpn_last_1y LONG,
    redeem_digital_cpn_last_2y LONG,
    redeem_digital_cpn_last_3y LONG,
    redeem_clp_cpn_last_1y LONG,
    redeem_clp_cpn_last_2y LONG,
    redeem_clp_cpn_last_3y LONG,
    RUN_DATE DATE,
    DATASET_CD STRING
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gm_scores} (
    MBRSHP_SID LONG,
    mbrshp_nbr LONG,
    score DECIMAL(5,4),
    decile INTEGER,
    score_date DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {digital_propensity_features} (
    MBRSHP_SID LONG,
    AVERAGE_DAYS_BETWEEN_LAST_180DAYS DOUBLE,
    BEFORE_LAST_Q60DAYS_DAYTRIPS LONG,
    AVERAGE_DAYS_BETWEEN_LAST_60DAYS DOUBLE,
    LAST_60DAYS_PERISHABLES_TRIPS LONG,
    LAST_60DAYS_GROCERY_TRIPS LONG,
    `LAST_TWENTY-SIX_WEEK_DISTINCT_DAYS` LONG,
    BEFORE_LAST_Q60DAYS_BASKETSIZE DOUBLE,
    L52W_MEDIAN_BASKETSIZE FLOAT,
    BEFORE_LAST_Q60DAYS_SALES DOUBLE,
    BEFORE_LAST_QUARTER_BASKETSIZE DOUBLE,
    BEFORE_LAST_QUARTER_SALES DOUBLE,
    L52W_G4W_STDEV_SPEND DOUBLE,
    LAST_60DAYS_INSTORE_BASKETSIZE DOUBLE,
    LAST_60DAYS_SALES DOUBLE,
    LATEST_MBRSHP_NBR STRING,
    FISCAL_WEEK_END DATE
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {digital_trips_scores} (
    MBRSHP_SID LONG,
    FISCAL_WEEK_END DATE,
    LATEST_MBRSHP_NBR STRING,
    trips_score FLOAT
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {digital_sales_scores} (
    MBRSHP_SID LONG,
    FISCAL_WEEK_END DATE,
    LATEST_MBRSHP_NBR STRING,
    sales_score FLOAT
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {pipeline_summary_report} (
    run_date DATE NOT NULL,
    job_name STRING NOT NULL,
    start_time STRING,
    end_time STRING,
    duration_in_hs DOUBLE,
    delta_to_prev_week STRING,
    URL_of_job_run STRING    
)
USING DELTA
TBLPROPERTIES (
    'predictiveOptimization'='true',
    'delta.autoOptimize.optimizeWrite'='true',
    'delta.autoOptimize.autoCompact'='true',
    'delta.enableChangeDataFeed'='true',
    'delta.columnMapping.mode'='name'
)
""")